<a href="https://colab.research.google.com/github/RasheedBlake/rb_gpt/blob/main/gpt_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pickle
# import argparse

"""
parser = argparse.ArgumentParser(description='This is a demo program')
parser.add_argument('-batch_size', type=str, required=True, help="Please provide a batch_size")
args = parser.parse_args()
print(f'batch size: {args.batch_size}')
"""
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

batch_size    = 64
block_size    = 256
max_iters     = 12000
learning_rate = 2.5e-4
eval_iters    = 500
n_embd        = 512
n_head        = 8
n_layer       = 12
dropout       = 0.10








cuda


In [ ]:
chars = ""
with open("drive/MyDrive/Amazon Review LLM Data/train.csv", "r", encoding="utf-8") as f:
    text = f.read()
    chars = sorted(list(set(text)))

vocab_size = len(chars)

In [ ]:
string_to_int = {ch:i for i, ch in enumerate(chars)}
int_to_string = {i:ch for i, ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

In [ ]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:

"""
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('when input is', context, 'target is', target)
"""

In [ ]:
# assumes globals exist: n_embd, n_head, n_layer, block_size, dropout, vocab_size, device, decode

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # causal mask sized to maximum context (block_size)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        # mask to current T; tril is already a buffer on the correct device
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


class FeedFoward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),                    # fixed capitalization and enabled
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x + y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        return x


class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.block_size = block_size  # remember maximum context length
        self.token_embedding_table   = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(self.block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        # clamp context to last block_size tokens to avoid pos-embed overflow
        if index.size(1) > self.block_size:
            index = index[:, -self.block_size:]
        B, T = index.shape

        tok_emb = self.token_embedding_table(index)                       # (B,T,C)
        pos = torch.arange(T, device=index.device)                        # (T,)
        pos_emb = self.position_embedding_table(pos)                      # (T,C)
        x = tok_emb + pos_emb                                             # (B,T,C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                          # (B,T,vocab)

        loss = None
        if targets is not None:
            if targets.size(1) > self.block_size:
                targets = targets[:, -self.block_size:]
            # flatten for CE
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    def generate(self, index, max_new_tokens):
        self.eval()
        idx = index
        with torch.no_grad():
            for _ in range(max_new_tokens):
                # only feed the last block_size tokens to the model
                logits, _ = self.forward(idx[:, -self.block_size:])
                logits = logits[:, -1, :]                 # (B, vocab)
                probs = F.softmax(logits, dim=-1)         # (B, vocab)
                idx_next = torch.multinomial(probs, num_samples=1)  # (B,1)
                idx = torch.cat((idx, idx_next), dim=1)   # (B, T+1)
        return idx


# ---- usage ----
model = GPTLanguageModel(vocab_size)

"""
print('loading model parameters...')
with open ('drive/MyDrive/Amazon Review LLM Data/model-02.pkl', 'rb') as f:
  model = pickle.load(f)
print('loaded successfully')

m = model.to(device)
"""

#context = torch.zeros((1, 1), dtype=torch.long, device=device)
#generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
#print(generated_chars)

In [ ]:
run = 0
while run <= 0:

  print('loading model parameters...')
  with open ('drive/MyDrive/Amazon Review LLM Data/model-02.pkl', 'rb') as f:
    model = pickle.load(f)
  print('loaded successfully')

  m = model.to(device)

  optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

  for iter in range(max_iters):
      if iter % eval_iters ==0:
          losses = estimate_loss()
          print(f"step: {iter}, train loss: {losses['train']:.3f} val loss: {losses['val']:.3f}")
          diff = losses['train'] - losses['val']
          print(f"Difference: {diff:.3f}")
      xb, yb = get_batch('train')

      logits, loss = model.forward(xb, yb)
      optimizer.zero_grad(set_to_none=True)
      loss.backward()
      optimizer.step()
  print(loss.item())

  if  diff <= -1.0:
      print("❌ Stopping training early due to overfitting.")
      run = 2500

  with open('drive/MyDrive/Amazon Review LLM Data/model-02.pkl', 'wb') as f:
    pickle.dump(model, f)
  print("model saved")
  run += 1
  print(f"run: {run}")
  print("--" * 10)
print("done")


In [ ]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)

In [ ]:
#version 2

import torch
import torch.nn as nn
import torch.nn.functional as F

# --------------------
# Device & hyperparams
# --------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

batch_size    = 64
block_size    = 256
max_iters     = 12000
learning_rate = 2.5e-4
eval_iters    = 500
n_embd        = 512
n_head        = 8
n_layer       = 12
dropout       = 0.10

CKPT_PATH = 'drive/MyDrive/Amazon Review LLM Data/model-02.pt'
OVERFIT_GAP = 1.0      # stop if val >= train + 1.0
PATIENCE = 5           # eval checkpoints without improvement before early stop

# --------------------
# Data: char-level
# --------------------
with open("drive/MyDrive/Amazon Review LLM Data/train.csv", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)

string_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_string = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    src = train_data if split == 'train' else val_data
    ix = torch.randint(len(src) - block_size, (batch_size,))
    x = torch.stack([src[i:i + block_size] for i in ix])
    y = torch.stack([src[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters, device=device)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# --------------------
# Model (GPT-like)
# --------------------
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):  # renamed from FeedFoward
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x + y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(self.block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        if index.size(1) > self.block_size:
            index = index[:, -self.block_size:]
        B, T = index.shape
        tok_emb = self.token_embedding_table(index)       # (B, T, C)
        pos = torch.arange(T, device=index.device)        # (T,)
        pos_emb = self.position_embedding_table(pos)      # (T, C)
        x = tok_emb + pos_emb                             # (B, T, C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                          # (B, T, vocab)

        loss = None
        if targets is not None:
            if targets.size(1) > self.block_size:
                targets = targets[:, -self.block_size:]
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, index, max_new_tokens):
        self.eval()
        idx = index
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# --------------------
# Init / (optional) resume from checkpoint
# --------------------
model = GPTLanguageModel(vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

start_iter = 0
best_val_loss = float('inf')

try:
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    start_iter = ckpt.get('iter', 0)
    best_val_loss = ckpt.get('best_val_loss', float('inf'))
    print(f"Loaded checkpoint from {CKPT_PATH} @ iter={start_iter}, best_val={best_val_loss:.4f}")
except FileNotFoundError:
    print("No existing checkpoint found; training from scratch.")

# --------------------
# Training loop with true early stopping & best checkpointing
# --------------------
no_improve = 0
final_loss = None

for iter in range(start_iter, max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        train_loss = losses['train']
        val_loss = losses['val']
        diff = train_loss - val_loss
        print(f"step: {iter}, train loss: {train_loss:.3f}  val loss: {val_loss:.3f}  (diff train-val: {diff:.3f})")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'iter': iter,
                'best_val_loss': best_val_loss,
                'vocab': {'chars': chars},
            }, CKPT_PATH)
            no_improve = 0
            print(f"✅ Saved new best checkpoint: val {best_val_loss:.4f}")
        else:
            no_improve += 1

        if diff <= -OVERFIT_GAP:
            print("❌ Early stop: validation loss exceeds training loss by >= "
                  f"{OVERFIT_GAP:.2f} (likely overfitting).")
            break

        if no_improve >= PATIENCE:
            print(f" Early stop: no validation improvement for {PATIENCE} evals.")
            break

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    final_loss = float(loss.item())

print("done. last minibatch loss:", None if final_loss is None else f"{final_loss:.4f}")
print(f"Best validation loss observed: {best_val_loss:.4f}")
print(f"Best checkpoint saved to: {CKPT_PATH}")

# --------------------
# Text generation using the best checkpoint
# --------------------
with torch.no_grad():
    try:
        ckpt = torch.load(CKPT_PATH, map_location=device)
        model.load_state_dict(ckpt['model_state'])
        model.eval()
        # Your requested snippet (adapted to 'model' instead of 'm'):
        context = torch.zeros((1, 1), dtype=torch.long, device=device)
        generated_chars = decode(model.generate(context, max_new_tokens=500)[0].tolist())
        print(generated_chars)
    except FileNotFoundError:
        print("No checkpoint found for generation.")


cuda
Loaded checkpoint from drive/MyDrive/Amazon Review LLM Data/model-02.pt @ iter=6000, best_val=1.1299
step: 6000, train loss: 1.133  val loss: 1.128  (diff train-val: 0.005)
✅ Saved new best checkpoint: val 1.1282
step: 6500, train loss: 1.121  val loss: 1.121  (diff train-val: -0.000)
✅ Saved new best checkpoint: val 1.1211
step: 7000, train loss: 1.116  val loss: 1.113  (diff train-val: 0.004)
✅ Saved new best checkpoint: val 1.1127
step: 7500, train loss: 1.110  val loss: 1.106  (diff train-val: 0.004)
✅ Saved new best checkpoint: val 1.1056
step: 8000, train loss: 1.100  val loss: 1.098  (diff train-val: 0.002)
✅ Saved new best checkpoint: val 1.0977
step: 8500, train loss: 1.098  val loss: 1.094  (diff train-val: 0.004)
✅ Saved new best checkpoint: val 1.0942
step: 9000, train loss: 1.093  val loss: 1.089  (diff train-val: 0.004)
✅ Saved new best checkpoint: val 1.0888
step: 9500, train loss: 1.085  val loss: 1.083  (diff train-val: 0.002)
✅ Saved new best checkpoint: val 1.08

In [ ]:
#version 3

import torch
import torch.nn as nn
import torch.nn.functional as F
import os

# --------------------
# Device & hyperparams
# --------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

batch_size    = 12
block_size    = 768
max_iters     = 150_000
learning_rate = 2.5e-4
eval_iters    = 700
n_embd        = 640
n_head        = 10
n_layer       = 12
dropout       = 0.08



CKPT_PATH = 'drive/MyDrive/txt data/models/model-01.pt'
FINAL_CKPT_PATH = 'drive/MyDrive/txt data/models/model-01-final.pt'
OVERFIT_GAP = 1.0      # stop if val >= train + 1.0
PATIENCE = 5           # eval checkpoints without improvement before early stop

os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)

# --------------------
# Data: char-level
# --------------------
with open("drive/MyDrive/txt data/combined data/all_text_files.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)

string_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_string = {i: ch for i, ch in enumerate(chars)}

def encode(s): return [string_to_int[c] for c in s]
def decode(l): return ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    src = train_data if split == 'train' else val_data
    ix = torch.randint(len(src) - block_size, (batch_size,))
    x = torch.stack([src[i:i + block_size] for i in ix])
    y = torch.stack([src[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters, device=device)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# --------------------
# Model (GPT-like)
# --------------------
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x + y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(self.block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        if index.size(1) > self.block_size:
            index = index[:, -self.block_size:]
        B, T = index.shape
        tok_emb = self.token_embedding_table(index)       # (B, T, C)
        pos = torch.arange(T, device=index.device)        # (T,)
        pos_emb = self.position_embedding_table(pos)      # (T, C)
        x = tok_emb + pos_emb                             # (B, T, C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                          # (B, T, vocab)

        loss = None
        if targets is not None:
            if targets.size(1) > self.block_size:
                targets = targets[:, -self.block_size:]
            B2, T2, C = logits.shape
            loss = F.cross_entropy(logits.view(B2 * T2, C), targets.view(B2 * T2))
        return logits, loss

    @torch.no_grad()
    def generate(self, index, max_new_tokens):
        self.eval()
        idx = index
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# --------------------
# Helpers: meta + saving
# --------------------
def build_meta():
    return {
        'stoi': string_to_int,
        'itos': int_to_string,
        'vocab_size': len(string_to_int),
        # helpful for sanity on load:
        'block_size': block_size,
        'n_embd': n_embd,
        'n_head': n_head,
        'n_layer': n_layer,
    }

def save_ckpt(path, iter_idx, best_val):
    torch.save({
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'iter': iter_idx,
        'best_val_loss': best_val,
        'meta': build_meta(),   # <<<<<< always include meta
    }, path)

# --------------------
# Optional resume (read meta first, if present)
# --------------------
start_iter = 0
best_val_loss = float('inf')
ckpt = None

try:
    ckpt = torch.load(CKPT_PATH, map_location=device)
    if 'meta' in ckpt:
        meta = ckpt['meta']
        # Override tokenizer + vocab_size from meta to ensure consistency
        string_to_int = meta['stoi']
        int_to_string = meta['itos']
        vocab_size = meta['vocab_size']
        # refresh encode/decode to use restored mappings
        def encode(s): return [string_to_int[c] for c in s]
        def decode(l): return ''.join([int_to_string[i] for i in l])
        print(f"Loaded meta from {CKPT_PATH} (vocab_size={vocab_size})")
except FileNotFoundError:
    pass  # training from scratch

# --------------------
# Init model/opt (after potential meta override)
# --------------------
model = GPTLanguageModel(vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

if ckpt is not None:
    try:
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        start_iter = ckpt.get('iter', 0)
        best_val_loss = ckpt.get('best_val_loss', float('inf'))
        print(f"Resumed from {CKPT_PATH} @ iter={start_iter}, best_val={best_val_loss:.4f}")
    except Exception as e:
        print(f"Warning: failed to load weights/optimizer from checkpoint: {e}")
        print("Continuing with fresh model/optimizer.")

# --------------------
# Training loop with early stopping & best checkpointing
# --------------------
no_improve = 0
final_loss = None

for iter in range(start_iter, max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        train_loss = losses['train']
        val_loss = losses['val']
        diff = train_loss - val_loss
        print(f"step: {iter}, train loss: {train_loss:.3f}  val loss: {val_loss:.3f}  (diff train-val: {diff:.3f})")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_ckpt(CKPT_PATH, iter, best_val_loss)
            no_improve = 0
            print(f"✅ Saved new best checkpoint (with meta): val {best_val_loss:.4f}")
        else:
            no_improve += 1

        if diff <= -OVERFIT_GAP:
            print("❌ Early stop: validation loss exceeds training loss by "
                  f">= {OVERFIT_GAP:.2f} (likely overfitting).")
            break

        if no_improve >= PATIENCE:
            print(f" Early stop: no validation improvement for {PATIENCE} evals.")
            break

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    final_loss = float(loss.item())

print("done. last minibatch loss:", None if final_loss is None else f"{final_loss:.4f}")
print(f"Best validation loss observed: {best_val_loss:.4f}")
print(f"Best checkpoint saved to: {CKPT_PATH}")

# Always save a final snapshot with meta as well
save_ckpt(FINAL_CKPT_PATH, iter, best_val_loss)
print(f"Final checkpoint (with meta) saved to: {FINAL_CKPT_PATH}")

# --------------------
# Text generation using the final/best checkpoint (reload meta)
# --------------------
with torch.no_grad():
    try:
        ckpt = torch.load(CKPT_PATH, map_location=device)
    except FileNotFoundError:
        ckpt = torch.load(FINAL_CKPT_PATH, map_location=device)

    meta = ckpt.get('meta', None)
    if meta is None:
        raise RuntimeError("Checkpoint missing 'meta'—expected after this run.")

    string_to_int = meta['stoi']
    int_to_string = meta['itos']
    def encode(s): return [string_to_int[c] for c in s]
    def decode(l): return ''.join(int_to_string[i] for i in l)

    # Rebuild model if vocab changed (safety)
    if model.lm_head.out_features != meta['vocab_size']:
        vocab_size = meta['vocab_size']
        model = GPTLanguageModel(vocab_size).to(device)

    model.load_state_dict(ckpt['model_state'])
    model.eval()

    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    generated_chars = decode(model.generate(context, max_new_tokens=500)[0].tolist())
    print(generated_chars)


cuda


KeyboardInterrupt: 

In [ ]:
#version 4 (parquet)
# train_parquet_streaming_char.py
import os, glob, random, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast  # modern AMP API

# =========================
# Device & training config
# =========================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)

# Model / training hyperparams (fits 40GB GPU comfortably)
batch_size       = 32
grad_accum_steps = 2                 # effective batch ≈ 64
block_size       = 512               # lower to 256 if you want more batch
max_iters        = 300_000
learning_rate    = 2.5e-4
eval_interval    = 1200
eval_iters       = 800
n_embd           = 512
n_head           = 8
n_layer          = 12
dropout          = 0.10

adam_betas       = (0.9, 0.95)
weight_decay     = 0.10
grad_clip        = 1.0
use_amp          = True

OVERFIT_GAP      = 1.0
PATIENCE         = 5

# =========================
# Checkpoints (YOUR PATHS)
# =========================
CKPT_PATH        = 'drive/MyDrive/english_dataset/Models/model-02.pt'
FINAL_CKPT_PATH  = 'drive/MyDrive/english_dataset/Models/model-02-final.pt'
os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)

# =========================
# Parquet input (STREAMING) — YOUR GLOB
# =========================
DATA_DIR_OR_GLOB = "drive/MyDrive/english_dataset/data files/*.parquet"
TEXT_COLUMN      = "text"
SHUFFLE_FILES    = True            # shuffle file order each cycle
ROW_SHUFFLE      = True            # shuffle rows within each batch (if pyarrow)
JOIN_SEP         = "\n"            # used when stitching row text

# Streaming buffers (bigger; you have plenty of RAM)
RING_TOKENS_CAP  = 20_000_000      # ~20M int32 ≈ 80 MB
VAL_TOKENS_CAP   = 4_000_000       # ~4M int64  ≈ 32 MB
WARMUP_TOKENS    = 2 * batch_size * (block_size + 1)

# ================
# Utilities: IO
# ================
def list_parquet_files(path_or_glob):
    if os.path.isdir(path_or_glob):
        files = glob.glob(os.path.join(path_or_glob, "*.parquet"))
    else:
        files = glob.glob(path_or_glob, recursive=True)
    if SHUFFLE_FILES:
        random.shuffle(files)
    if not files:
        raise FileNotFoundError(f"No parquet files matched: {path_or_glob}")
    return files

def iter_text_rows(files, text_col):
    """Yield lists of strings (row batches) without loading everything at once."""
    # Prefer pyarrow streaming
    try:
        import pyarrow.dataset as ds
        for fchunk in files:
            dataset = ds.dataset([fchunk])
            scanner = ds.Scanner.from_dataset(dataset, columns=[text_col])
            for batch in scanner.to_batches():
                arr = batch.column(0).to_pylist()
                if ROW_SHUFFLE:
                    random.shuffle(arr)
                rows = [str(x) for x in arr if x is not None and str(x)]
                if rows:
                    yield rows
    except Exception:
        # Fallback to pandas
        import pandas as pd
        for p in files:
            df = pd.read_parquet(p, columns=[text_col])
            s = df[text_col].dropna().astype(str).tolist()
            if ROW_SHUFFLE:
                random.shuffle(s)
            CH = 50_000
            for i in range(0, len(s), CH):
                chunk = s[i:i+CH]
                if chunk:
                    yield chunk

# =========================
# Phase 1: Build vocabulary
# =========================
def build_vocab(files, text_col):
    charset = set()
    for rows in iter_text_rows(files, text_col):
        for t in rows:
            charset.update(t)
    charset.add("\n")
    chars = sorted(list(charset))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for i, ch in enumerate(chars)}
    return chars, stoi, itos

# =========================
# Streaming ring buffer
# =========================
class TokenRing:
    def __init__(self, capacity_tokens):
        self.capacity = int(capacity_tokens)
        self.buf = np.zeros(self.capacity, dtype=np.int32)  # compact
        self.write = 0
        self.filled = 0

    def add_many(self, token_list):
        """Robust wrap-safe append (handles huge chunks too)."""
        t = np.asarray(token_list, dtype=np.int32)
        n = int(t.shape[0])
        if n == 0:
            return
        # If incoming chunk is larger than capacity, keep only the tail
        if n >= self.capacity:
            self.buf[:] = t[-self.capacity:]
            self.write = 0
            self.filled = self.capacity
            return

        a = self.write
        end = a + n
        if end <= self.capacity:
            # contiguous write
            self.buf[a:end] = t
        else:
            # wrapped write
            first = self.capacity - a
            self.buf[a:] = t[:first]
            self.buf[:end - self.capacity] = t[first:]

        self.write = end % self.capacity
        self.filled = min(self.capacity, self.filled + n)

    def available(self):
        return self.filled

    def sample_batch(self, batch_size, block_size):
        if self.filled < block_size + 1:
            raise RuntimeError("Ring buffer underflow; not enough tokens.")

        if self.filled < self.capacity:
            hi = self.filled - (block_size + 1)
            starts = np.random.randint(0, hi + 1, size=batch_size, dtype=np.int64)
            x = np.stack([self.buf[s:s+block_size] for s in starts], axis=0)
            y = np.stack([self.buf[s+1:s+block_size+1] for s in starts], axis=0)
        else:
            w = self.write
            seg1_len = max(0, w - (block_size + 1))
            seg2_len = max(0, (self.capacity - (block_size + 1)) - w)
            if seg1_len + seg2_len <= 0:
                raise RuntimeError("No valid sampling positions (block_size too large).")
            p1 = seg1_len / (seg1_len + seg2_len)
            n1 = np.random.binomial(batch_size, p1)
            n2 = batch_size - n1
            s1 = np.random.randint(0, seg1_len + 1, size=n1, dtype=np.int64) if n1>0 else np.array([], dtype=np.int64)
            s2 = np.random.randint(w, w + seg2_len + 1, size=n2, dtype=np.int64) if n2>0 else np.array([], dtype=np.int64)
            starts = np.concatenate([s1, s2], axis=0)

            def take(start):
                end = start + block_size + 1
                if end <= self.capacity:
                    return self.buf[start:end]
                k = end - self.capacity
                return np.concatenate([self.buf[start:], self.buf[:k]], axis=0)

            windows = np.stack([take(s) for s in starts], axis=0)
            x = windows[:, :-1]
            y = windows[:, 1:]

        x = torch.from_numpy(x.astype(np.int64)).to(device)
        y = torch.from_numpy(y.astype(np.int64)).to(device)
        return x, y

# =========================
# Model (GPT-like)
# =========================
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x); q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1); wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = self.ln1(x + self.sa(x))
        x = self.ln2(x + self.ffwd(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(self.block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    def forward(self, idx, targets=None):
        if idx.size(1) > self.block_size:
            idx = idx[:, -self.block_size:]
        B, T = idx.shape
        tok = self.token_embedding_table(idx)
        pos = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok + pos
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            if targets.size(1) > self.block_size:
                targets = targets[:, -self.block_size:]
            B2, T2, C = logits.shape
            loss = F.cross_entropy(logits.view(B2*T2, C), targets.view(B2*T2))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        self.eval()
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            nxt = torch.multinomial(probs, 1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

# =========================
# Meta & checkpoint helpers
# =========================
def build_meta(stoi, itos):
    return {
        'stoi': stoi,
        'itos': itos,
        'vocab_size': len(stoi),
        'block_size': block_size,
        'n_embd': n_embd,
        'n_head': n_head,
        'n_layer': n_layer,
        'source': 'parquet_streaming',
        'text_column': TEXT_COLUMN,
    }

def save_ckpt(path, model, optimizer, iter_idx, best_val, meta):
    torch.save({
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'iter': iter_idx,
        'best_val_loss': best_val,
        'meta': meta,
    }, path)

# =========================
# Build vocab (1 pass)
# =========================
files = list_parquet_files(DATA_DIR_OR_GLOB)
chars, stoi, itos = build_vocab(files, TEXT_COLUMN)
vocab_size = len(chars)
print(f"vocab_size: {vocab_size}")

# =========================
# Init model/opt/AMP + resume
# =========================
model = GPTLanguageModel(vocab_size).to(device)
scaler = GradScaler(device="cuda", enabled=(use_amp and device=='cuda'))
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate,
                              betas=adam_betas, weight_decay=weight_decay)

start_iter = 0
best_val_loss = float('inf')
resume_meta = None
ckpt = None
try:
    ckpt = torch.load(CKPT_PATH, map_location=device)
    resume_meta = ckpt.get('meta', None)
except FileNotFoundError:
    pass

if ckpt is not None:
    if resume_meta is not None:
        for k in ('block_size','n_embd','n_head','n_layer'):
            if k in resume_meta and resume_meta[k] != globals()[k]:
                raise ValueError(f"{k} mismatch: ckpt={resume_meta[k]} vs code={globals()[k]}")
        stoi = resume_meta['stoi']; itos = resume_meta['itos']
        if len(stoi) != vocab_size:
            vocab_size = len(stoi)
            model = GPTLanguageModel(vocab_size).to(device)
    try:
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        start_iter = int(ckpt.get('iter', 0))
        best_val_loss = float(ckpt.get('best_val_loss', float('inf')))
        print(f"Resumed from {CKPT_PATH} @ iter={start_iter}, best_val={best_val_loss:.4f}")
    except Exception as e:
        print("Warning: failed to load weights/optimizer; starting fresh.", repr(e))

# =========================
# Build small validation & training ring
# =========================
ring = TokenRing(RING_TOKENS_CAP)
val_tokens = []

def encode_text(s):
    return [stoi[ch] for ch in (s + "\n") if ch in stoi]

def feed_until_warm(files):
    global val_tokens
    for rows in iter_text_rows(files, TEXT_COLUMN):
        for t in rows:
            ids = encode_text(t)
            if len(val_tokens) < VAL_TOKENS_CAP:
                val_tokens.extend(ids)
            else:
                ring.add_many(ids)
        if ring.available() >= WARMUP_TOKENS and len(val_tokens) >= (block_size + 1) * 1024:
            return

feed_until_warm(files)
if ring.available() < WARMUP_TOKENS:
    feed_until_warm(files)

val_arr = np.array(val_tokens, dtype=np.int64)
del val_tokens
print(f"val buffer tokens: {len(val_arr):,} ; ring filled: {ring.available():,}")

def get_val_batch():
    hi = len(val_arr) - (block_size + 1)
    idxs = np.random.randint(0, hi + 1, size=batch_size, dtype=np.int64)
    x = np.stack([val_arr[s:s+block_size] for s in idxs], axis=0)
    y = np.stack([val_arr[s+1:s+block_size+1] for s in idxs], axis=0)
    return torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)

@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = {}
    tl = []
    for _ in range(eval_iters):
        xb, yb = ring.sample_batch(batch_size, block_size)
        _, loss = model(xb, yb)
        tl.append(loss.item())
    losses['train'] = float(np.mean(tl))
    vl = []
    for _ in range(eval_iters):
        xb, yb = get_val_batch()
        _, loss = model(xb, yb)
        vl.append(loss.item())
    losses['val'] = float(np.mean(vl))
    model.train()
    return losses

# =========================
# Training loop (stream refill)
# =========================
no_improve = 0
final_loss = None
last_iter = start_iter
meta = build_meta(stoi, itos)

def infinite_file_cycle():
    while True:
        fs = list_parquet_files(DATA_DIR_OR_GLOB)
        for f in fs:
            yield f

file_iter = infinite_file_cycle()

def refill_ring(target_tokens=RING_TOKENS_CAP // 4):
    added = 0
    while added < target_tokens:
        f = next(file_iter)
        for rows in iter_text_rows([f], TEXT_COLUMN):
            for t in rows:
                ids = encode_text(t)
                ring.add_many(ids)
                added += len(ids)
                if added >= target_tokens:
                    return

for iter in range(start_iter, max_iters):
    last_iter = iter

    if iter % eval_interval == 0:
        losses = estimate_loss()
        train_loss = losses['train']; val_loss = losses['val']
        diff = train_loss - val_loss
        print(f"step {iter}: train {train_loss:.3f} | val {val_loss:.3f} | diff {diff:.3f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_ckpt(CKPT_PATH, model, optimizer, iter, best_val_loss, meta)
            no_improve = 0
            print(f"✅ Saved new best (val {best_val_loss:.4f})")
        else:
            no_improve += 1

        if diff <= -OVERFIT_GAP:
            print("❌ Early stop: validation worse than train by >= gap.")
            break
        if no_improve >= PATIENCE:
            print(f" Early stop: no val improvement for {PATIENCE} evals.")
            break

    if ring.available() < WARMUP_TOKENS * 2:
        refill_ring()

    optimizer.zero_grad(set_to_none=True)
    accum = 0.0
    for _ in range(grad_accum_steps):
        xb, yb = ring.sample_batch(batch_size, block_size)
        with autocast(device_type="cuda", enabled=(use_amp and device=='cuda')):
            _, loss = model(xb, yb)
            loss = loss / grad_accum_steps
        scaler.scale(loss).backward()
        accum += float(loss.item())

    if grad_clip is not None:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    scaler.step(optimizer)
    scaler.update()
    final_loss = accum

print("done. last (normalized) minibatch loss:", None if final_loss is None else f"{final_loss:.4f}")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Best checkpoint: {CKPT_PATH}")

save_ckpt(FINAL_CKPT_PATH, model, optimizer, last_iter, best_val_loss, meta)
print(f"Final checkpoint (with meta): {FINAL_CKPT_PATH}")

# =========================
# Quick sample
# =========================
with torch.no_grad():
    try:
        ck = torch.load(CKPT_PATH, map_location=device)
    except FileNotFoundError:
        ck = torch.load(FINAL_CKPT_PATH, map_location=device)
    meta2 = ck['meta']
    stoi2 = meta2['stoi']; itos2 = meta2['itos']; vs2 = meta2['vocab_size']
    if model.lm_head.out_features != vs2:
        model = GPTLanguageModel(vs2).to(device)
    model.load_state_dict(ck['model_state']); model.eval()
    ctx = torch.zeros((1, 1), dtype=torch.long, device=device)
    out = model.generate(ctx, max_new_tokens=200)[0].tolist()
    print(''.join(itos2[i] for i in out))


device: cuda
vocab_size: 389
val buffer tokens: 4,168,062 ; ring filled: 20,000,000
step 0: train 6.261 | val 6.287 | diff -0.025
✅ Saved new best (val 6.2865)
step 1200: train 1.460 | val 1.442 | diff 0.018
✅ Saved new best (val 1.4425)
step 2400: train 1.263 | val 1.300 | diff -0.036
✅ Saved new best (val 1.2996)
step 3600: train 1.189 | val 1.253 | diff -0.065
✅ Saved new best (val 1.2533)
step 4800: train 1.138 | val 1.231 | diff -0.092
✅ Saved new best (val 1.2307)
step 6000: train 1.100 | val 1.213 | diff -0.113
✅ Saved new best (val 1.2131)
step 7200: train 1.069 | val 1.208 | diff -0.138
✅ Saved new best (val 1.2076)
step 8400: train 1.047 | val 1.195 | diff -0.148
✅ Saved new best (val 1.1947)
step 9600: train 1.022 | val 1.192 | diff -0.170
✅ Saved new best (val 1.1919)
step 10800: train 1.003 | val 1.193 | diff -0.190
step 12000: train 0.983 | val 1.193 | diff -0.210
step 13200: train 0.969 | val 1.189 | diff -0.221
✅ Saved new best (val 1.1894)
step 14400: train 0.954 | val

In [ ]:
#version 5 (tsv)

import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

# --------------------
# Device, seeds & hyperparams
# --------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

torch.manual_seed(1337)
if device == 'cuda':
    torch.cuda.manual_seed_all(1337)

batch_size    = 16
block_size    = 1024
max_iters     = 200_000
learning_rate = 2.5e-4
eval_iters    = 800
n_embd        = 768
n_head        = 12
n_layer       = 16
dropout       = 0.05
chunksize     = 100_000  # number of rows per text chunk

CKPT_PATH       = 'drive/MyDrive/Kaggle Amazon/models/model-01.pt'
FINAL_CKPT_PATH = 'drive/MyDrive/Kaggle Amazon/models/model-01-final.pt'
file_path       = 'drive/MyDrive/Kaggle Amazon/data/*.tsv'
OVERFIT_GAP     = 1.0      # stop if val >= train + 1.0
PATIENCE        = 5         # eval checkpoints without improvement before early stop

os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)

# --------------------
# Utilities: read corpus
# --------------------
def load_corpus_tsvs(pattern: str, chunksize: int) -> str:
    corpus = []
    files = sorted(glob.glob(pattern))
    if not files:
        raise RuntimeError(f"No data files match pattern: {pattern}")
    for file in files:
        print(f"Reading file: {file}")
        for df_chunk in pd.read_csv(file, sep="\t", usecols=["review_body"], chunksize=chunksize):
            s = df_chunk["review_body"].fillna("").astype(str).str.cat(sep="\n")
            corpus.append(s)
    text_str = "\n".join(corpus)
    if not text_str:
        raise RuntimeError("Corpus is empty after reading TSVs.")
    return text_str

# --------------------
# Optional resume: try to read checkpoint/meta FIRST
# --------------------
ckpt = None
meta = None
try:
    ckpt = torch.load(CKPT_PATH, map_location=device)
    meta = ckpt.get('meta', None)
    if meta is not None:
        print(f"Loaded meta from {CKPT_PATH} (vocab_size={meta.get('vocab_size')})")
except FileNotFoundError:
    pass

# --------------------
# Build corpus string
# --------------------
text_str = load_corpus_tsvs(file_path, chunksize)

# --------------------
# Tokenizer (char-level) — resume-safe
# --------------------
if meta is not None:
    string_to_int = meta['stoi']
    int_to_string = meta['itos']
    vocab_size    = meta['vocab_size']

    # On resume: filter out chars that aren't in the saved vocab (no <unk> token here)
    def encode(s): return [string_to_int[c] for c in s if c in string_to_int]
    def decode(l): return ''.join(int_to_string[i] for i in l if i in int_to_string)
else:
    chars = sorted(list(set(text_str)))
    vocab_size = len(chars)
    string_to_int = {ch: i for i, ch in enumerate(chars)}
    int_to_string = {i: ch for i, ch in enumerate(chars)}
    def encode(s): return [string_to_int[c] for c in s]
    def decode(l): return ''.join(int_to_string[i] for i in l)

# Encode data with the *active* vocab (fresh or from meta)
data = torch.tensor(encode(text_str), dtype=torch.long)
if len(data) <= block_size + 1:
    raise RuntimeError(f"Not enough encoded data ({len(data)}) for block_size={block_size}")

# Train/val split
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f"Dataset chars: {len(data):,} | train: {len(train_data):,} | val: {len(val_data):,}")

# --------------------
# Batching
# --------------------
def get_batch(split):
    src = train_data if split == 'train' else val_data
    if len(src) <= block_size:
        raise RuntimeError(f"Not enough data ({len(src)}) for block_size={block_size}")
    ix = torch.randint(0, len(src) - block_size - 1, (batch_size,))
    x = torch.stack([src[i:i + block_size] for i in ix])
    y = torch.stack([src[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)  # keep on CPU
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = float(losses.mean().item())
    model.train()
    return out

# --------------------
# Model (GPT-like)
# --------------------
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('mask', torch.tril(torch.ones(block_size, block_size, dtype=torch.bool)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = (q @ k.transpose(-2, -1)) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(~self.mask[:T, :T], float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),                 # (GELU is a quality bump, but keeping your ReLU)
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x + y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(self.block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        # Clip to block_size if longer
        if index.size(1) > self.block_size:
            index = index[:, -self.block_size:]

        B, T = index.shape
        tok_emb = self.token_embedding_table(index)        # (B, T, C)
        pos = torch.arange(T, device=index.device)         # (T,)
        pos_emb = self.position_embedding_table(pos)       # (T, C)
        x = tok_emb + pos_emb                              # (B, T, C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                           # (B, T, vocab)

        loss = None
        if targets is not None:
            if targets.size(1) > self.block_size:
                targets = targets[:, -self.block_size:]
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1)
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, index, max_new_tokens):
        self.eval()
        idx = index
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# --------------------
# Helpers: meta + saving
# --------------------
def build_meta():
    return {
        'stoi': string_to_int,
        'itos': int_to_string,
        'vocab_size': len(string_to_int),
        'block_size': block_size,
        'n_embd': n_embd,
        'n_head': n_head,
        'n_layer': n_layer,
    }

def save_ckpt(path, iter_idx, best_val):
    torch.save({
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'iter': iter_idx,
        'best_val_loss': best_val,
        'meta': build_meta(),   # always include meta
    }, path)

# --------------------
# Init model/optimizer (with correct vocab_size)
# --------------------
model = GPTLanguageModel(vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Attempt to load weights/opt state if we had found a ckpt
start_iter = 0
best_val_loss = float('inf')
if ckpt is not None:
    try:
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        start_iter = ckpt.get('iter', 0)
        best_val_loss = ckpt.get('best_val_loss', float('inf'))
        print(f"Resumed from {CKPT_PATH} @ iter={start_iter}, best_val={best_val_loss:.4f}")
    except Exception as e:
        print(f"Warning: failed to load weights/optimizer from checkpoint: {e}")
        print("Continuing with fresh model/optimizer.")

# --------------------
# Training loop with early stopping & best checkpointing
# --------------------
no_improve = 0
final_loss = None
current_iter = start_iter

for iter_idx in range(start_iter, max_iters):
    current_iter = iter_idx

    if iter_idx % eval_iters == 0:
        losses = estimate_loss()
        train_loss = losses['train']
        val_loss = losses['val']
        diff = train_loss - val_loss
        print(f"step: {iter_idx}, train loss: {train_loss:.3f}  val loss: {val_loss:.3f}  (diff train-val: {diff:.3f})")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_ckpt(CKPT_PATH, iter_idx, best_val_loss)
            no_improve = 0
            print(f"✅ Saved new best checkpoint (with meta): val {best_val_loss:.4f}")
        else:
            no_improve += 1

        if diff <= -OVERFIT_GAP:
            print(f"❌ Early stop: validation loss exceeds training loss by >= {OVERFIT_GAP:.2f} (likely overfitting).")
            break

        if no_improve >= PATIENCE:
            print(f" Early stop: no validation improvement for {PATIENCE} evals.")
            break

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    final_loss = float(loss.item())

print("done. last minibatch loss:", None if final_loss is None else f"{final_loss:.4f}")
print(f"Best validation loss observed: {best_val_loss:.4f}")
print(f"Best checkpoint saved to: {CKPT_PATH}")

# Always save a final snapshot with meta as well
save_ckpt(FINAL_CKPT_PATH, current_iter, best_val_loss)
print(f"Final checkpoint (with meta) saved to: {FINAL_CKPT_PATH}")

# --------------------
# Text generation using the final/best checkpoint (reload meta)
# --------------------
with torch.no_grad():
    try:
        ckpt_gen = torch.load(CKPT_PATH, map_location=device)
    except FileNotFoundError:
        ckpt_gen = torch.load(FINAL_CKPT_PATH, map_location=device)

    meta_gen = ckpt_gen.get('meta', None)
    if meta_gen is None:
        raise RuntimeError("Checkpoint missing 'meta'—expected after this run.")

    stoi = meta_gen['stoi']
    itos = meta_gen['itos']
    def decode_gen(l): return ''.join(itos[i] for i in l if i in itos)

    # Rebuild model if vocab changed
    if model.lm_head.out_features != meta_gen['vocab_size']:
        model = GPTLanguageModel(meta_gen['vocab_size']).to(device)

    model.load_state_dict(ckpt_gen['model_state'])
    model.eval()

    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    generated_chars = decode_gen(model.generate(context, max_new_tokens=500)[0].tolist())
    print(generated_chars)


cuda
Reading file: drive/MyDrive/Kaggle Amazon/data/amazon_reviews_multilingual_US_v1_00.tsv
Reading file: drive/MyDrive/Kaggle Amazon/data/amazon_reviews_us_Apparel_v1_00.tsv
Reading file: drive/MyDrive/Kaggle Amazon/data/amazon_reviews_us_Automotive_v1_00.tsv
Reading file: drive/MyDrive/Kaggle Amazon/data/amazon_reviews_us_Baby_v1_00.tsv
Reading file: drive/MyDrive/Kaggle Amazon/data/amazon_reviews_us_Beauty_v1_00.tsv
Reading file: drive/MyDrive/Kaggle Amazon/data/amazon_reviews_us_Books_v1_02.tsv
Reading file: drive/MyDrive/Kaggle Amazon/data/amazon_reviews_us_Camera_v1_00.tsv
Reading file: drive/MyDrive/Kaggle Amazon/data/amazon_reviews_us_Digital_Ebook_Purchase_v1_01.tsv
Reading file: drive/MyDrive/Kaggle Amazon/data/amazon_reviews_us_Digital_Music_Purchase_v1_00.tsv
Reading file: drive/MyDrive/Kaggle Amazon/data/amazon_reviews_us_Digital_Software_v1_00.tsv
Reading file: drive/MyDrive/Kaggle Amazon/data/amazon_reviews_us_Digital_Video_Download_v1_00.tsv
Reading file: drive/MyDrive

In [ ]:
import os, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# --------------------
# Device & hyperparams (yours)
# --------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

batch_size    = 12
block_size    = 768
max_iters     = 150_000
learning_rate = 2.5e-4
eval_iters    = 700
n_embd        = 640
n_head        = 10
n_layer       = 12
dropout       = 0.08

CKPT_PATH = 'drive/MyDrive/txt data/models/model-02.pt'
FINAL_CKPT_PATH = 'drive/MyDrive/txt data/models/model-02-final.pt'
TEXT_PATH = "drive/MyDrive/txt data/combined data/all_text_files.txt"

PATIENCE    = 5          # patience-only early stopping
WEIGHT_DECAY = 0.10      # AdamW weight decay
GRAD_CLIP   = 1.0        # gradient clipping (norm)
EMA_DECAY   = 0.999      # EMA decay
WARMUP_STEPS = min(3_000, max(100, max_iters // 10))  # cosine warmup

os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
os.makedirs(os.path.dirname(FINAL_CKPT_PATH), exist_ok=True)

# --------------------
# Streaming buffers (tiny vs your RAM)
# --------------------
RING_TOKENS_CAP = 20_000_000   # ~80 MB as int32
VAL_TOKENS_CAP  = 2_000_000    # ~16 MB as int64
WARMUP_TOKENS   = 2 * batch_size * (block_size + 1)

# --------------------
# Stream file (no giant strings)
# --------------------
def stream_text_chunks(path, chunk_size=1<<20):  # ~1 MB
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        while True:
            s = f.read(chunk_size)
            if not s:
                break
            yield s

# One pass to build vocabulary (tiny memory)
charset = set()
for chunk in stream_text_chunks(TEXT_PATH):
    charset.update(chunk)
charset.add("\n")
chars = sorted(list(charset))
vocab_size = len(chars)

string_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_string = {i: ch for i, ch in enumerate(chars)}
def encode(s):  return [string_to_int.get(c) for c in s if c in string_to_int]
def decode(ids): return ''.join(int_to_string[i] for i in ids)

print(f"vocab_size: {vocab_size}")

# --------------------
# Ring buffer for training tokens
# --------------------
class TokenRing:
    def __init__(self, capacity_tokens):
        self.capacity = int(capacity_tokens)
        self.buf = np.zeros(self.capacity, dtype=np.int32)
        self.write = 0
        self.filled = 0

    def add_many(self, token_list):
        """Robust, wrap-safe append; trims if chunk > capacity."""
        t = np.asarray(token_list, dtype=np.int32)
        n = int(t.shape[0])
        if n == 0:
            return
        if n >= self.capacity:
            self.buf[:] = t[-self.capacity:]
            self.write = 0
            self.filled = self.capacity
            return
        a = self.write
        end = a + n
        if end <= self.capacity:
            self.buf[a:end] = t
        else:
            first = self.capacity - a
            self.buf[a:] = t[:first]
            self.buf[:end - self.capacity] = t[first:]
        self.write = end % self.capacity
        self.filled = min(self.capacity, self.filled + n)

    def available(self):
        return self.filled

    def sample_batch(self, B, T):
        if self.filled < T + 1:
            raise RuntimeError("Ring underflow; not enough tokens.")
        if self.filled < self.capacity:
            hi = self.filled - (T + 1)
            starts = np.random.randint(0, hi + 1, size=B, dtype=np.int64)
            x = np.stack([self.buf[s:s+T] for s in starts], axis=0)
            y = np.stack([self.buf[s+1:s+T+1] for s in starts], axis=0)
        else:
            w = self.write
            seg1_len = max(0, w - (T + 1))
            seg2_len = max(0, (self.capacity - (T + 1)) - w)
            if seg1_len + seg2_len <= 0:
                raise RuntimeError("No valid sample positions.")
            p1 = seg1_len / (seg1_len + seg2_len)
            n1 = np.random.binomial(B, p1)
            n2 = B - n1
            s1 = np.random.randint(0, seg1_len + 1, size=n1, dtype=np.int64) if n1>0 else np.empty(0, np.int64)
            s2 = np.random.randint(w, w + seg2_len + 1, size=n2, dtype=np.int64) if n2>0 else np.empty(0, np.int64)
            starts = np.concatenate([s1, s2], axis=0)

            def take(start):
                end = start + T + 1
                if end <= self.capacity:
                    return self.buf[start:end]
                k = end - self.capacity
                return np.concatenate([self.buf[start:], self.buf[:k]], axis=0)

            windows = np.stack([take(s) for s in starts], axis=0)
            x = windows[:, :-1]; y = windows[:, 1:]

        x = torch.from_numpy(x.astype(np.int64)).to(device)
        y = torch.from_numpy(y.astype(np.int64)).to(device)
        return x, y

ring = TokenRing(RING_TOKENS_CAP)
val_tokens = []

def feed_until_warm():
    global val_tokens
    for chunk in stream_text_chunks(TEXT_PATH):
        ids = encode(chunk + "\n")
        if not ids:
            continue
        if len(val_tokens) < VAL_TOKENS_CAP:
            need = VAL_TOKENS_CAP - len(val_tokens)
            if len(ids) <= need:
                val_tokens.extend(ids)
            else:
                val_tokens.extend(ids[:need])
                ring.add_many(ids[need:])
        else:
            ring.add_many(ids)
        if ring.available() >= WARMUP_TOKENS and len(val_tokens) >= (block_size + 1) * 1024:
            break

feed_until_warm()
if ring.available() < WARMUP_TOKENS:
    feed_until_warm()

val_arr = np.array(val_tokens, dtype=np.int64); del val_tokens
print(f"val buffer tokens: {len(val_arr):,} ; ring filled: {ring.available():,}")

def get_val_batch():
    hi = len(val_arr) - (block_size + 1)
    idxs = np.random.randint(0, hi + 1, size=batch_size, dtype=np.int64)
    x = np.stack([val_arr[s:s+block_size] for s in idxs], axis=0)
    y = np.stack([val_arr[s+1:s+block_size+1] for s in idxs], axis=0)
    return torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)

# --------------------
# Model (GPT-like) — UPDATED
# --------------------
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x); q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj  = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    # accepts n_head as a proper keyword (fix 1)
    def __init__(self, n_embd, n_head, **_ignore):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x); x = self.ln1(x + y)
        y = self.ffwd(x); x = self.ln2(x + y)
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size_):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table    = nn.Embedding(vocab_size_, n_embd)
        self.position_embedding_table = nn.Embedding(self.block_size, n_embd)
        # call Block with positional args (also works if older Block took n_head_ etc.) (fix 2)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size_)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        if index.size(1) > self.block_size:
            index = index[:, -self.block_size:]
        B, T = index.shape
        tok = self.token_embedding_table(index)
        pos = torch.arange(T, device=index.device)
        pos_emb = self.position_embedding_table(pos)
        x = tok + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            if targets.size(1) > self.block_size:
                targets = targets[:, -self.block_size:]
            B2, T2, C = logits.shape
            loss = F.cross_entropy(logits.view(B2*T2, C), targets.view(B2*T2))
        return logits, loss

    @torch.no_grad()
    def generate(self, index, max_new_tokens):
        self.eval()
        idx = index
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            nxt = torch.multinomial(probs, 1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


# --------------------
# Meta + saving
# --------------------
def build_meta():
    return {
        'stoi': string_to_int,
        'itos': int_to_string,
        'vocab_size': len(string_to_int),
        'block_size': block_size,
        'n_embd': n_embd, 'n_head': n_head, 'n_layer': n_layer,
    }

def atomic_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

def save_ckpt(path, iter_idx, best_val, model_state, ema_state, optimizer_state):
    atomic_save({
        'model_state': model_state,
        'ema_state': ema_state,
        'optimizer_state': optimizer_state,
        'iter': iter_idx,
        'best_val_loss': best_val,
        'meta': build_meta(),
    }, path)

# --------------------
# Optional resume (tokenizer only first)
# --------------------
start_iter = 0
best_val_loss = float('inf')
ckpt = None
try:
    ckpt = torch.load(CKPT_PATH, map_location=device)
    if 'meta' in ckpt:
        meta = ckpt['meta']
        string_to_int = meta['stoi']; int_to_string = meta['itos']
        vocab_size = meta['vocab_size']
        def encode(s):  return [string_to_int.get(c) for c in s if c in string_to_int]
        def decode(ids): return ''.join(int_to_string[i] for i in ids)
        print(f"Loaded meta from {CKPT_PATH} (vocab_size={vocab_size})")
except FileNotFoundError:
    pass

# --------------------
# Init model/EMA/opt/sched
# --------------------
model = GPTLanguageModel(vocab_size).to(device)
ema_model = GPTLanguageModel(vocab_size).to(device)
ema_model.load_state_dict(model.state_dict())
for p in ema_model.parameters():
    p.requires_grad_(False)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=WEIGHT_DECAY)

def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    p = (step - WARMUP_STEPS) / max(1, max_iters - WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * p))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

if ckpt is not None:
    try:
        model.load_state_dict(ckpt['model_state'])
        if 'ema_state' in ckpt:
            ema_model.load_state_dict(ckpt['ema_state'])
        else:
            ema_model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        start_iter = ckpt.get('iter', 0)
        best_val_loss = ckpt.get('best_val_loss', float('inf'))
        print(f"Resumed from {CKPT_PATH} @ iter={start_iter}, best_val={best_val_loss:.4f}")
    except Exception as e:
        print(f"Warning: failed to load weights/optimizer from checkpoint: {e}")
        ema_model.load_state_dict(model.state_dict())

# --------------------
# Eval (uses EMA)
# --------------------
@torch.no_grad()
def estimate_loss(eval_model):
    out = {}
    eval_model.eval()
    # train loss from ring
    tl = []
    for _ in range(eval_iters):
        X, Y = ring.sample_batch(batch_size, block_size)
        _, loss = eval_model(X, Y)
        tl.append(loss.item())
    out['train'] = float(np.mean(tl))
    # val loss from fixed buffer
    vl = []
    for _ in range(eval_iters):
        X, Y = get_val_batch()
        _, loss = eval_model(X, Y)
        vl.append(loss.item())
    out['val'] = float(np.mean(vl))
    model.train()
    return out

def ema_update():
    with torch.no_grad():
        msd = model.state_dict()
        esd = ema_model.state_dict()
        for k in msd.keys():
            esd[k].mul_(EMA_DECAY).add_(msd[k], alpha=1.0 - EMA_DECAY)

# --------------------
# Training (samples from ring; patience-only early stop)
# --------------------
no_improve = 0
final_loss = None
last_iter = start_iter

for iter in range(start_iter, max_iters):
    last_iter = iter

    # Periodic eval with EMA weights
    if iter % eval_iters == 0:
        losses = estimate_loss(ema_model)
        train_loss = losses['train']; val_loss = losses['val']
        diff = train_loss - val_loss
        print(f"step: {iter}, train {train_loss:.3f}  val {val_loss:.3f}  (diff {diff:+.3f})")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_ckpt(CKPT_PATH, iter, best_val_loss,
                      model.state_dict(), ema_model.state_dict(), optimizer.state_dict())
            no_improve = 0
            print(f"✅ Saved new best (EMA) with meta: val {best_val_loss:.4f}")
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"⏹️ Early stop: no validation improvement for {PATIENCE} evals.")
            break

    # Train step
    optimizer.zero_grad(set_to_none=True)
    xb, yb = ring.sample_batch(batch_size, block_size)
    logits, loss = model(xb, yb)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()
    scheduler.step()
    ema_update()
    final_loss = float(loss.item())

print("done. last minibatch loss:", None if final_loss is None else f"{final_loss:.4f}")
print(f"Best validation loss observed: {best_val_loss:.4f}")
print(f"Best checkpoint saved to: {CKPT_PATH}")

save_ckpt(FINAL_CKPT_PATH, last_iter, best_val_loss,
          model.state_dict(), ema_model.state_dict(), optimizer.state_dict())
print(f"Final checkpoint (EMA + meta) saved to: {FINAL_CKPT_PATH}")

# --------------------
# Text generation (reload best EMA)
# --------------------
with torch.no_grad():
    try:
        ckpt = torch.load(CKPT_PATH, map_location=device)
    except FileNotFoundError:
        ckpt = torch.load(FINAL_CKPT_PATH, map_location=device)
    meta = ckpt['meta']
    string_to_int = meta['stoi']; int_to_string = meta['itos']
    if vocab_size != meta['vocab_size']:
        vocab_size = meta['vocab_size']
    gen_model = GPTLanguageModel(vocab_size).to(device)
    state = ckpt.get('ema_state', ckpt['model_state'])
    gen_model.load_state_dict(state)
    gen_model.eval()
    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    print(decode(gen_model.generate(context, max_new_tokens=500)[0].tolist()))


cuda
vocab_size: 31336
val buffer tokens: 2,000,000 ; ring filled: 97,154
step: 0, train 10.396  val 10.399  (diff -0.003)
✅ Saved new best (EMA) with meta: val 10.3993
step: 700, train 5.175  val 5.196  (diff -0.021)
✅ Saved new best (EMA) with meta: val 5.1963
step: 1400, train 2.694  val 2.736  (diff -0.042)
✅ Saved new best (EMA) with meta: val 2.7361
step: 2100, train 2.469  val 2.643  (diff -0.174)
✅ Saved new best (EMA) with meta: val 2.6430
step: 2800, train 1.444  val 2.706  (diff -1.262)
step: 3500, train 0.092  val 3.076  (diff -2.983)


KeyboardInterrupt: 

In [ ]:
import os, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# --------------------
# Device & hyperparams (yours)
# --------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

batch_size    = 12
block_size    = 768
max_iters     = 120_000
learning_rate = 2.5e-4
eval_iters    = 700
n_embd        = 640
n_head        = 10
n_layer       = 12
dropout       = 0.08

CKPT_PATH = 'drive/MyDrive/txt data/models/model-04.pt'
FINAL_CKPT_PATH = 'drive/MyDrive/txt data/models/model-04-final.pt'
TEXT_PATH = "drive/MyDrive/txt data/combined data/all_text_files.txt"

PATIENCE     = 5          # patience-only early stopping
WEIGHT_DECAY = 0.10       # AdamW weight decay
GRAD_CLIP    = 1.0        # gradient clipping (norm)
EMA_DECAY    = 0.999      # EMA decay
WARMUP_STEPS = min(3_000, max(100, max_iters // 10))  # cosine warmup

os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
os.makedirs(os.path.dirname(FINAL_CKPT_PATH), exist_ok=True)

# --------------------
# Streaming buffers (tiny vs your RAM)
# --------------------
RING_TOKENS_CAP = 20_000_000   # ~80 MB as int32
VAL_TOKENS_CAP  = 2_000_000    # ~16 MB as int64
WARMUP_TOKENS   = 2 * batch_size * (block_size + 1)

# --------------------
# Stream file (no giant strings)
# --------------------
def stream_text_chunks(path, chunk_size=1<<20):  # ~1 MB
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        while True:
            s = f.read(chunk_size)
            if not s:
                break
            yield s

# One pass to build vocabulary (tiny memory)
charset = set()
for chunk in stream_text_chunks(TEXT_PATH):
    charset.update(chunk)
charset.add("\n")
chars = sorted(list(charset))
vocab_size = len(chars)

string_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_string = {i: ch for i, ch in enumerate(chars)}
def encode(s):  return [string_to_int[c] for c in s if c in string_to_int]
def decode(ids): return ''.join(int_to_string[i] for i in ids)

print(f"vocab_size: {vocab_size}")

# --------------------
# Ring buffer for training tokens
# --------------------
class TokenRing:
    def __init__(self, capacity_tokens):
        self.capacity = int(capacity_tokens)
        self.buf = np.zeros(self.capacity, dtype=np.int32)
        self.write = 0
        self.filled = 0

    def add_many(self, token_list):
        """Robust, wrap-safe append; trims if chunk > capacity."""
        t = np.asarray(token_list, dtype=np.int32)
        n = int(t.shape[0])
        if n == 0:
            return
        if n >= self.capacity:
            self.buf[:] = t[-self.capacity:]
            self.write = 0
            self.filled = self.capacity
            return
        a = self.write
        end = a + n
        if end <= self.capacity:
            self.buf[a:end] = t
        else:
            first = self.capacity - a
            self.buf[a:] = t[:first]
            self.buf[:end - self.capacity] = t[first:]
        self.write = end % self.capacity
        self.filled = min(self.capacity, self.filled + n)

    def available(self):
        return self.filled

    def sample_batch(self, B, T):
        if self.filled < T + 1:
            raise RuntimeError("Ring underflow; not enough tokens.")
        if self.filled < self.capacity:
            hi = self.filled - (T + 1)
            starts = np.random.randint(0, hi + 1, size=B, dtype=np.int64)
            x = np.stack([self.buf[s:s+T] for s in starts], axis=0)
            y = np.stack([self.buf[s+1:s+T+1] for s in starts], axis=0)
        else:
            w = self.write
            seg1_len = max(0, w - (T + 1))
            seg2_len = max(0, (self.capacity - (T + 1)) - w)
            if seg1_len + seg2_len <= 0:
                raise RuntimeError("No valid sample positions.")
            p1 = seg1_len / (seg1_len + seg2_len)
            n1 = np.random.binomial(B, p1)
            n2 = B - n1
            s1 = np.random.randint(0, seg1_len + 1, size=n1, dtype=np.int64) if n1>0 else np.empty(0, np.int64)
            s2 = np.random.randint(w, w + seg2_len + 1, size=n2, dtype=np.int64) if n2>0 else np.empty(0, np.int64)
            starts = np.concatenate([s1, s2], axis=0)

            def take(start):
                end = start + T + 1
                if end <= self.capacity:
                    return self.buf[start:end]
                k = end - self.capacity
                return np.concatenate([self.buf[start:], self.buf[:k]], axis=0)

            windows = np.stack([take(s) for s in starts], axis=0)
            x = windows[:, :-1]; y = windows[:, 1:]

        x = torch.from_numpy(x.astype(np.int64)).to(device)
        y = torch.from_numpy(y.astype(np.int64)).to(device)
        return x, y

ring = TokenRing(RING_TOKENS_CAP)

# --------------------
# Build val buffer + prefill ring to CAPACITY, then keep an iterator for top-ups
# --------------------
val_tokens = []

# First pass: fill validation up to cap; overflow into ring
for chunk in stream_text_chunks(TEXT_PATH):
    ids = encode(chunk + "\n")
    if not ids:
        continue
    if len(val_tokens) < VAL_TOKENS_CAP:
        need = VAL_TOKENS_CAP - len(val_tokens)
        take = min(need, len(ids))
        val_tokens.extend(ids[:take])
        leftover = ids[take:]
        if leftover:
            ring.add_many(leftover)
    else:
        ring.add_many(ids)

# Iterator for subsequent refills
_chunk_iter = iter(stream_text_chunks(TEXT_PATH))

def refill_ring_to(target_tokens):
    """Keep reading stream until ring.available() >= target_tokens (wraps file if needed)."""
    global _chunk_iter
    while ring.available() < target_tokens:
        try:
            s = next(_chunk_iter)
        except StopIteration:
            _chunk_iter = iter(stream_text_chunks(TEXT_PATH))
            s = next(_chunk_iter)
        ring.add_many(encode(s + "\n"))

# Ensure we can sample, then fill the ring all the way
if ring.available() < WARMUP_TOKENS:
    refill_ring_to(WARMUP_TOKENS)
refill_ring_to(RING_TOKENS_CAP)

# Finalize and shuffle validation buffer to reduce positional bias
val_arr = np.array(val_tokens, dtype=np.int64); del val_tokens
np.random.default_rng().shuffle(val_arr)

print(f"val buffer tokens: {len(val_arr):,} ; ring filled: {ring.available():,}")

def get_val_batch():
    hi = len(val_arr) - (block_size + 1)
    idxs = np.random.randint(0, hi + 1, size=batch_size, dtype=np.int64)
    x = np.stack([val_arr[s:s+block_size] for s in idxs], axis=0)
    y = np.stack([val_arr[s+1:s+block_size+1] for s in idxs], axis=0)
    return torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)

# Trickle fresh tokens during training so the model doesn't memorize a tiny pool
TOKENS_PER_TOPUP = 2 * batch_size * (block_size + 1)
def top_up_ring(min_add_tokens=TOKENS_PER_TOPUP):
    global _chunk_iter
    added = 0
    while added < min_add_tokens:
        try:
            s = next(_chunk_iter)
        except StopIteration:
            _chunk_iter = iter(stream_text_chunks(TEXT_PATH))
            s = next(_chunk_iter)
        ids = encode(s + "\n")
        if not ids:
            continue
        ring.add_many(ids)
        added += len(ids)

# --------------------
# Model (GPT-like)
# --------------------
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x); q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1); wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj  = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd_):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd_, 4 * n_embd_),
            nn.ReLU(),
            nn.Linear(4 * n_embd_, n_embd_),
            nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd_, n_head_):
        super().__init__()
        head_size = n_embd_ // n_head_
        self.sa = MultiHeadAttention(n_head_, head_size)
        self.ffwd = FeedForward(n_embd_)
        self.ln1 = nn.LayerNorm(n_embd_)
        self.ln2 = nn.LayerNorm(n_embd_)
    def forward(self, x):
        y = self.sa(x); x = self.ln1(x + y)
        y = self.ffwd(x); x = self.ln2(x + y)
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size_):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table    = nn.Embedding(vocab_size_, n_embd)
        self.position_embedding_table = nn.Embedding(self.block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size_)
        self.apply(self._init_weights)
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    def forward(self, index, targets=None):
        if index.size(1) > self.block_size:
            index = index[:, -self.block_size:]
        B, T = index.shape
        tok = self.token_embedding_table(index)
        pos = torch.arange(T, device=index.device)
        pos_emb = self.position_embedding_table(pos)
        x = tok + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            if targets.size(1) > self.block_size:
                targets = targets[:, -self.block_size:]
            B2, T2, C = logits.shape
            loss = F.cross_entropy(logits.view(B2*T2, C), targets.view(B2*T2))
        return logits, loss
    @torch.no_grad()
    def generate(self, index, max_new_tokens):
        self.eval()
        idx = index
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            nxt = torch.multinomial(probs, 1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

# --------------------
# Meta + saving
# --------------------
def build_meta():
    return {
        'stoi': string_to_int,
        'itos': int_to_string,
        'vocab_size': len(string_to_int),
        'block_size': block_size,
        'n_embd': n_embd, 'n_head': n_head, 'n_layer': n_layer,
    }

def atomic_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

def save_ckpt(path, iter_idx, best_val, model_state, ema_state, optimizer_state):
    atomic_save({
        'model_state': model_state,
        'ema_state': ema_state,
        'optimizer_state': optimizer_state,
        'iter': iter_idx,
        'best_val_loss': best_val,
        'meta': build_meta(),
    }, path)

# --------------------
# Optional resume (tokenizer only first)
# --------------------
start_iter = 0
best_val_loss = float('inf')
ckpt = None
try:
    ckpt = torch.load(CKPT_PATH, map_location=device)
    if 'meta' in ckpt:
        meta = ckpt['meta']
        string_to_int = meta['stoi']; int_to_string = meta['itos']
        vocab_size = meta['vocab_size']
        def encode(s):  return [string_to_int[c] for c in s if c in string_to_int]
        def decode(ids): return ''.join(int_to_string[i] for i in ids)
        print(f"Loaded meta from {CKPT_PATH} (vocab_size={vocab_size})")
except FileNotFoundError:
    pass

# --------------------
# Init model/EMA/opt/sched
# --------------------
model = GPTLanguageModel(vocab_size).to(device)
ema_model = GPTLanguageModel(vocab_size).to(device)
ema_model.load_state_dict(model.state_dict())
for p in ema_model.parameters():
    p.requires_grad_(False)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=WEIGHT_DECAY)

def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    p = (step - WARMUP_STEPS) / max(1, max_iters - WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * p))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

if ckpt is not None:
    try:
        model.load_state_dict(ckpt['model_state'])
        if 'ema_state' in ckpt:
            ema_model.load_state_dict(ckpt['ema_state'])
        else:
            ema_model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        start_iter = ckpt.get('iter', 0)
        best_val_loss = ckpt.get('best_val_loss', float('inf'))
        print(f"Resumed from {CKPT_PATH} @ iter={start_iter}, best_val={best_val_loss:.4f}")
    except Exception as e:
        print(f"Warning: failed to load weights/optimizer from checkpoint: {e}")
        ema_model.load_state_dict(model.state_dict())

# --------------------
# Eval (uses EMA)
# --------------------
@torch.no_grad()
def estimate_loss(eval_model):
    out = {}
    eval_model.eval()
    # train loss from ring
    tl = []
    for _ in range(eval_iters):
        X, Y = ring.sample_batch(batch_size, block_size)
        _, loss = eval_model(X, Y)
        tl.append(loss.item())
    out['train'] = float(np.mean(tl))
    # val loss from fixed buffer
    vl = []
    for _ in range(eval_iters):
        X, Y = get_val_batch()
        _, loss = eval_model(X, Y)
        vl.append(loss.item())
    out['val'] = float(np.mean(vl))
    model.train()
    return out

def ema_update():
    with torch.no_grad():
        msd = model.state_dict()
        esd = ema_model.state_dict()
        for k in msd.keys():
            esd[k].mul_(EMA_DECAY).add_(msd[k], alpha=1.0 - EMA_DECAY)

# --------------------
# Training (samples from ring; patience-only early stop)
# --------------------
no_improve = 0
final_loss = None
last_iter = start_iter

for iter in range(start_iter, max_iters):
    last_iter = iter

    # Keep the training pool fresh every N steps
    if iter % 100 == 0:
        top_up_ring()

    # Periodic eval with EMA weights
    if iter % eval_iters == 0:
        losses = estimate_loss(ema_model)
        train_loss = losses['train']; val_loss = losses['val']
        diff = train_loss - val_loss
        print(f"step: {iter}, train {train_loss:.3f}  val {val_loss:.3f}  (diff {diff:+.3f})")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_ckpt(CKPT_PATH, iter, best_val_loss,
                      model.state_dict(), ema_model.state_dict(), optimizer.state_dict())
            no_improve = 0
            print(f"✅ Saved new best (EMA) with meta: val {best_val_loss:.4f}")
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"⏹️ Early stop: no validation improvement for {PATIENCE} evals.")
            break

    # Train step
    optimizer.zero_grad(set_to_none=True)
    xb, yb = ring.sample_batch(batch_size, block_size)
    logits, loss = model(xb, yb)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()
    scheduler.step()
    ema_update()
    final_loss = float(loss.item())

print("done. last minibatch loss:", None if final_loss is None else f"{final_loss:.4f}")
print(f"Best validation loss observed: {best_val_loss:.4f}")
print(f"Best checkpoint saved to: {CKPT_PATH}")

save_ckpt(FINAL_CKPT_PATH, last_iter, best_val_loss,
          model.state_dict(), ema_model.state_dict(), optimizer.state_dict())
print(f"Final checkpoint (EMA + meta) saved to: {FINAL_CKPT_PATH}")

# --------------------
# Text generation (reload best EMA)
# --------------------
with torch.no_grad():
    try:
        ckpt = torch.load(CKPT_PATH, map_location=device)
    except FileNotFoundError:
        ckpt = torch.load(FINAL_CKPT_PATH, map_location=device)
    meta = ckpt['meta']
    string_to_int = meta['stoi']; int_to_string = meta['itos']
    if vocab_size != meta['vocab_size']:
        vocab_size = meta['vocab_size']
    gen_model = GPTLanguageModel(vocab_size).to(device)
    state = ckpt.get('ema_state', ckpt['model_state'])
    gen_model.load_state_dict(state)
    gen_model.eval()
    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    print(decode(gen_model.generate(context, max_new_tokens=500)[0].tolist()))


cuda
vocab_size: 31336


KeyboardInterrupt: 

In [ ]:
import os, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import builtins  # <-- to avoid shadowing the built-in iter()

# --------------------
# Device, seeds & hyperparams (yours)
# --------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)

# Reproducibility & perf hints
torch.manual_seed(1337); np.random.seed(1337)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass

batch_size    = 12
block_size    = 1024   # ↑ longer context; reduce batch if you hit OOM
max_iters     = 400_000
learning_rate = 2.5e-4
eval_iters    = 150           # faster evals (tune 100–200 as you like)
n_embd        = 640
n_head        = 10
n_layer       = 12
dropout       = 0.05          # friendlier regularization

CKPT_PATH = 'drive/MyDrive/txt data/models/model-06.pt'
FINAL_CKPT_PATH = 'drive/MyDrive/txt data/models/model-06-final.pt'
TEXT_PATH = "drive/MyDrive/txt data/combined data/all_text_files.txt"

PATIENCE     = 1000              # patience-only early stopping
WEIGHT_DECAY = 0.01           # AdamW weight decay (common GPT default)
GRAD_CLIP    = 1.0            # gradient clipping (norm)
EMA_DECAY    = 0.999          # EMA decay
WARMUP_STEPS = min(3_000, max(100, max_iters // 10))  # cosine warmup

os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
os.makedirs(os.path.dirname(FINAL_CKPT_PATH), exist_ok=True)

# --------------------
# Streaming buffers (tiny vs your RAM)
# --------------------
RING_TOKENS_CAP = 200_000_000   # ~800 MB as int32 (you have 80 GB RAM)
VAL_TOKENS_CAP  = 2_000_000     # ~16 MB as int64
WARMUP_TOKENS   = 2 * batch_size * (block_size + 1)

# --------------------
# Stream file (no giant strings)
# --------------------
def stream_text_chunks(path, chunk_size=1<<20):  # ~1 MB
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        while True:
            s = f.read(chunk_size)
            if not s:
                break
            yield s

# --------------------
# Load tokenizer meta FIRST (if exists), else build from text
# --------------------
string_to_int, int_to_string, vocab_size = None, None, None

meta_ckpt = None
try:
    meta_ckpt = torch.load(CKPT_PATH, map_location='cpu')
except FileNotFoundError:
    meta_ckpt = None

if meta_ckpt is not None and 'meta' in meta_ckpt:
    meta = meta_ckpt['meta']
    string_to_int = meta['stoi']; int_to_string = meta['itos']
    vocab_size = meta['vocab_size']
    def encode(s):  return [string_to_int[c] for c in s if c in string_to_int]
    def decode(ids): return ''.join(int_to_string[i] for i in ids)
    print(f"Loaded tokenizer from {CKPT_PATH} (vocab_size={vocab_size})")
else:
    # Build from the text
    charset = set()
    for chunk in stream_text_chunks(TEXT_PATH):
        charset.update(chunk)
    charset.add("\n")
    chars = sorted(list(charset))
    vocab_size = len(chars)
    string_to_int = {ch: i for i, ch in enumerate(chars)}
    int_to_string = {i: ch for i, ch in enumerate(chars)}
    def encode(s):  return [string_to_int[c] for c in s if c in string_to_int]
    def decode(ids): return ''.join(int_to_string[i] for i in ids)
    print(f"Built tokenizer from text (vocab_size={vocab_size})")

print(f"vocab_size: {vocab_size}")

# --------------------
# Ring buffer for training tokens
# --------------------
class TokenRing:
    def __init__(self, capacity_tokens):
        self.capacity = int(capacity_tokens)
        self.buf = np.zeros(self.capacity, dtype=np.int32)
        self.write = 0
        self.filled = 0

    def add_many(self, token_list):
        """Robust, wrap-safe append; trims if chunk > capacity."""
        t = np.asarray(token_list, dtype=np.int32)
        n = int(t.shape[0])
        if n == 0:
            return
        if n >= self.capacity:
            self.buf[:] = t[-self.capacity:]
            self.write = 0
            self.filled = self.capacity
            return
        a = self.write
        end = a + n
        if end <= self.capacity:
            self.buf[a:end] = t
        else:
            first = self.capacity - a
            self.buf[a:] = t[:first]
            self.buf[:end - self.capacity] = t[first:]
        self.write = end % self.capacity
        self.filled = min(self.capacity, self.filled + n)

    def available(self):
        return self.filled

    def sample_batch(self, B, T):
        if self.filled < T + 1:
            raise RuntimeError("Ring underflow; not enough tokens.")
        if self.filled < self.capacity:
            hi = self.filled - (T + 1)
            starts = np.random.randint(0, hi + 1, size=B, dtype=np.int64)
            x = np.stack([self.buf[s:s+T] for s in starts], axis=0)
            y = np.stack([self.buf[s+1:s+T+1] for s in starts], axis=0)
        else:
            w = self.write
            seg1_len = max(0, w - (T + 1))
            seg2_len = max(0, (self.capacity - (T + 1)) - w)
            if seg1_len + seg2_len <= 0:
                raise RuntimeError("No valid sample positions.")
            p1 = seg1_len / (seg1_len + seg2_len) if (seg1_len + seg2_len) > 0 else 0.0
            n1 = np.random.binomial(B, p1)
            n2 = B - n1
            s1 = np.random.randint(0, seg1_len + 1, size=n1, dtype=np.int64) if n1>0 else np.empty(0, np.int64)
            s2 = np.random.randint(w, w + seg2_len + 1, size=n2, dtype=np.int64) if n2>0 else np.empty(0, np.int64)
            starts = np.concatenate([s1, s2], axis=0)

            def take(start):
                end = start + T + 1
                if end <= self.capacity:
                    return self.buf[start:end]
                k = end - self.capacity
                return np.concatenate([self.buf[start:], self.buf[:k]], axis=0)

            windows = np.stack([take(s) for s in starts], axis=0)
            x = windows[:, :-1]; y = windows[:, 1:]

        x = torch.from_numpy(x.astype(np.int64)).to(device)
        y = torch.from_numpy(y.astype(np.int64)).to(device)
        return x, y

ring = TokenRing(RING_TOKENS_CAP)

# --------------------
# Build val buffer + prefill ring to CAPACITY, then keep an iterator for top-ups
# --------------------
val_tokens = []

# First pass: fill validation up to cap; overflow into ring
for chunk in stream_text_chunks(TEXT_PATH):
    ids = encode(chunk + "\n")
    if not ids:
        continue
    if len(val_tokens) < VAL_TOKENS_CAP:
        need = VAL_TOKENS_CAP - len(val_tokens)
        take = min(need, len(ids))
        val_tokens.extend(ids[:take])
        leftover = ids[take:]
        if leftover:
            ring.add_many(leftover)
    else:
        ring.add_many(ids)

# Iterator for subsequent refills (use builtins.iter explicitly)
_chunk_iter = builtins.iter(stream_text_chunks(TEXT_PATH))

def refill_ring_to(target_tokens):
    """Keep reading stream until ring.available() >= target_tokens (wraps file if needed)."""
    global _chunk_iter
    while ring.available() < target_tokens:
        try:
            s = next(_chunk_iter)
        except StopIteration:
            _chunk_iter = builtins.iter(stream_text_chunks(TEXT_PATH))
            s = next(_chunk_iter)
        ring.add_many(encode(s + "\n"))

# Ensure we can sample, then fill the ring all the way
if ring.available() < WARMUP_TOKENS:
    refill_ring_to(WARMUP_TOKENS)
refill_ring_to(RING_TOKENS_CAP)

# Finalize validation buffer (DO NOT shuffle token order)
val_arr = np.array(val_tokens, dtype=np.int64); del val_tokens

print(f"val buffer tokens: {len(val_arr):,} ; ring filled: {ring.available():,}")

def get_val_batch():
    hi = len(val_arr) - (block_size + 1)
    idxs = np.random.randint(0, hi + 1, size=batch_size, dtype=np.int64)
    x = np.stack([val_arr[s:s+block_size] for s in idxs], axis=0)
    y = np.stack([val_arr[s+1:s+block_size+1] for s in idxs], axis=0)
    return torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)

# Aggressive churn across the dataset
TOKENS_PER_TOPUP = 100 * batch_size * (block_size + 1)

def top_up_ring(min_add_tokens=TOKENS_PER_TOPUP):
    global _chunk_iter
    added = 0
    while added < min_add_tokens:
        try:
            s = next(_chunk_iter)
        except StopIteration:
            _chunk_iter = builtins.iter(stream_text_chunks(TEXT_PATH))
            s = next(_chunk_iter)
        ids = encode(s + "\n")
        if not ids:
            continue
        ring.add_many(ids)
        added += len(ids)

# --------------------
# Model (GPT-like)
# --------------------
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x); q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1); wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj  = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd_):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd_, 4 * n_embd_),
            nn.ReLU(),
            nn.Linear(4 * n_embd_, n_embd_),
            nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd_, n_head_):
        super().__init__()
        head_size = n_embd_ // n_head_
        self.sa = MultiHeadAttention(n_head_, head_size)
        self.ffwd = FeedForward(n_embd_)
        self.ln1 = nn.LayerNorm(n_embd_)
        self.ln2 = nn.LayerNorm(n_embd_)
    def forward(self, x):
        y = self.sa(x); x = self.ln1(x + y)
        y = self.ffwd(x); x = self.ln2(x + y)
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size_):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table    = nn.Embedding(vocab_size_, n_embd)
        self.position_embedding_table = nn.Embedding(self.block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size_, bias=False)
        # Init
        self.apply(self._init_weights)
        # Tie weights
        self.lm_head.weight = self.token_embedding_table.weight

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if getattr(m, 'bias', None) is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        if index.size(1) > self.block_size:
            index = index[:, -self.block_size:]
        B, T = index.shape
        tok = self.token_embedding_table(index)
        pos = torch.arange(T, device=index.device)
        pos_emb = self.position_embedding_table(pos)
        x = tok + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            if targets.size(1) > self.block_size:
                targets = targets[:, -self.block_size:]
            B2, T2, C = logits.shape
            loss = F.cross_entropy(logits.view(B2*T2, C), targets.view(B2*T2))
        return logits, loss

    @torch.no_grad()
    def generate(self, index, max_new_tokens):
        self.eval()
        idx = index
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            nxt = torch.multinomial(probs, 1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

# --------------------
# Meta + saving
# --------------------
def build_meta():
    return {
        'stoi': string_to_int,
        'itos': int_to_string,
        'vocab_size': len(string_to_int),
        'block_size': block_size,
        'n_embd': n_embd, 'n_head': n_head, 'n_layer': n_layer,
    }

def atomic_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

def save_ckpt(path, iter_idx, best_val, model_state, ema_state, optimizer_state, scheduler_state):
    atomic_save({
        'model_state': model_state,
        'ema_state': ema_state,
        'optimizer_state': optimizer_state,
        'scheduler_state': scheduler_state,
        'iter': iter_idx,
        'best_val_loss': best_val,
        'meta': build_meta(),
    }, path)

# --------------------
# Init model/EMA/opt, optional resume (weights later)
# --------------------
model = GPTLanguageModel(vocab_size).to(device)
ema_model = GPTLanguageModel(vocab_size).to(device)
ema_model.load_state_dict(model.state_dict())
for p in ema_model.parameters():
    p.requires_grad_(False)

# Weight decay: exclude biases & LayerNorm from WD
decay, no_decay, seen = [], [], set()
for n, p in model.named_parameters():
    if id(p) in seen:
        continue
    seen.add(id(p))
    if p.ndim == 1 or n.endswith('.bias'):
        no_decay.append(p)
    else:
        decay.append(p)

optimizer = torch.optim.AdamW(
    [{'params': decay, 'weight_decay': WEIGHT_DECAY},
     {'params': no_decay, 'weight_decay': 0.0}],
    lr=learning_rate
)

# --- AMP setup (CUDA only) ---
use_amp = (device == 'cuda')
amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# LR schedule (warmup + cosine), respecting call order (scheduler.step AFTER optimizer.step)
def lr_lambda(step):
    step = step + 1  # because we call scheduler.step() after optimizer.step()
    if step <= WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    p = (step - WARMUP_STEPS) / max(1, max_iters - WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * p))

start_iter = 0
best_val_loss = float('inf')

# If full ckpt exists, resume
ckpt = None
try:
    ckpt = torch.load(CKPT_PATH, map_location=device)
except FileNotFoundError:
    ckpt = None

if ckpt is not None:
    try:
        model.load_state_dict(ckpt['model_state'])
        if 'ema_state' in ckpt:
            ema_model.load_state_dict(ckpt['ema_state'])
        else:
            ema_model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        start_iter = ckpt.get('iter', 0)
        best_val_loss = ckpt.get('best_val_loss', float('inf'))
        print(f"Resumed from {CKPT_PATH} @ iter={start_iter}, best_val={best_val_loss:.4f}")
    except Exception as e:
        print(f"Warning: failed to load weights/optimizer from checkpoint: {e}")
        ema_model.load_state_dict(model.state_dict())

# Scheduler created after knowing start_iter, then optionally load state
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda, last_epoch=start_iter - 1
)
if ckpt is not None and 'scheduler_state' in ckpt:
    try:
        scheduler.load_state_dict(ckpt['scheduler_state'])
    except Exception as e:
        print(f"Warning: failed to load scheduler state: {e}")

# Optional: compile for speed (PyTorch 2.x)
# We will handle the state dict prefix manually during load
try:
    model = torch.compile(model)
    # EMA model should not be compiled for state dict consistency
    # ema_model = torch.compile(ema_model)
except Exception:
    pass

# --------------------
# Eval (uses EMA)
# --------------------
@torch.no_grad()
def estimate_loss(eval_model):
    prev_mode = eval_model.training
    eval_model.eval()
    try:
        out = {}
        # train loss from ring
        tl = []
        for _ in range(eval_iters):
            X, Y = ring.sample_batch(batch_size, block_size)
            with torch.autocast(device_type='cuda', dtype=amp_dtype, enabled=use_amp):
                _, loss = eval_model(X, Y)
            tl.append(loss.item())
        out['train'] = float(np.mean(tl))
        # val loss from fixed buffer
        vl = []
        for _ in range(eval_iters):
            X, Y = get_val_batch()
            with torch.autocast(device_type='cuda', dtype=amp_dtype, enabled=use_amp):
                _, loss = eval_model(X, Y)
            vl.append(loss.item())
        out['val'] = float(np.mean(vl))
        return out
    finally:
        eval_model.train(prev_mode)

def ema_update():
    with torch.no_grad():
        msd = model.state_dict()
        esd = ema_model.state_dict()
        for k in msd.keys():
            # Handle potential _orig_mod prefix if the main model was compiled
            if k.startswith('_orig_mod.'):
                esd[k[len('_orig_mod.'):]].mul_(EMA_DECAY).add_(msd[k], alpha=1.0 - EMA_DECAY)
            else:
                esd[k].mul_(EMA_DECAY).add_(msd[k], alpha=1.0 - EMA_DECAY)

# --------------------
# Training (samples from ring; patience-only early stop)
# --------------------
no_improve = 0
final_loss = None
last_iter = start_iter

for step in range(start_iter, max_iters):   # <-- renamed from `for iter in ...`
    last_iter = step

    # Aggressively refresh training pool EVERY STEP
    top_up_ring()

    # Periodic eval with EMA weights
    if step % eval_iters == 0:              # <-- use step
        lr_now = scheduler.get_last_lr()[0]
        losses = estimate_loss(ema_model)
        train_loss = losses['train']; val_loss = losses['val']
        diff = train_loss - val_loss
        # also print bits-per-char (bpc)
        bpc_train = train_loss / math.log(2)
        bpc_val   = val_loss   / math.log(2)
        print(f"step: {step}, lr {lr_now:.6e}, "
              f"train {train_loss:.3f} (bpc {bpc_train:.3f})  "
              f"val {val_loss:.3f} (bpc {bpc_val:.3f})  "
              f"(diff {diff:+.3f})")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_ckpt(CKPT_PATH, step, best_val_loss,   # <-- pass step
                      model.state_dict(), ema_model.state_dict(),
                      optimizer.state_dict(), scheduler.state_dict())
            no_improve = 0
            print(f"✅ Saved new best (EMA) with meta: val {best_val_loss:.4f}")
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"⏹️ Early stop: no validation improvement for {PATIENCE} evals.")
            break

    # Train step (AMP)
    optimizer.zero_grad(set_to_none=True)
    xb, yb = ring.sample_batch(batch_size, block_size)
    with torch.autocast(device_type='cuda', dtype=amp_dtype, enabled=use_amp):
        logits, loss = model(xb, yb)

    scaler.scale(loss).backward()
    # Important: unscale before gradient clipping
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

    scaler.step(optimizer)
    scaler.update()

    scheduler.step()
    ema_update()
    final_loss = float(loss.item())

print("done. last minibatch loss:", None if final_loss is None else f"{final_loss:.4f}")
print(f"Best validation loss observed: {best_val_loss:.4f}")
print(f"Best checkpoint saved to: {CKPT_PATH}")

save_ckpt(FINAL_CKPT_PATH, last_iter, best_val_loss,
          model.state_dict(), ema_model.state_dict(),
          optimizer.state_dict(), scheduler.state_dict())
print(f"Final checkpoint (EMA + meta) saved to: {FINAL_CKPT_PATH}")

# --------------------
# Text generation (reload best EMA)
# --------------------
with torch.no_grad():
    try:
        ckpt = torch.load(CKPT_PATH, map_location=device)
    except FileNotFoundError:
        ckpt = torch.load(FINAL_CKPT_PATH, map_location=device)

    meta = ckpt['meta']
    string_to_int = meta['stoi']; int_to_string = meta['itos']
    # Rebuild the model for generation if vocab size changed
    if model.lm_head.out_features != meta['vocab_size']:
         vocab_size = meta['vocab_size']
         gen_model = GPTLanguageModel(vocab_size).to(device)
    else:
         gen_model = GPTLanguageModel(vocab_size).to(device)

    state = ckpt.get('ema_state', ckpt['model_state'])
    cleaned_state = {k.replace('_orig_mod.', ''): v for k, v in state.items()}
    gen_model.load_state_dict(cleaned_state)
    gen_model.eval()
    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    print(decode(gen_model.generate(context, max_new_tokens=500)[0].tolist()))


device: cuda
Built tokenizer from text (vocab_size=31336)
vocab_size: 31336
val buffer tokens: 2,000,000 ; ring filled: 200,000,000


/tmp/ipython-input-2068236219.py:401: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


step: 0, lr 8.333333e-08, train 10.359 (bpc 14.945)  val 10.395 (bpc 14.996)  (diff -0.036)
✅ Saved new best (EMA) with meta: val 10.3947
step: 150, lr 1.258333e-05, train 9.861 (bpc 14.227)  val 9.861 (bpc 14.227)  (diff -0.000)
✅ Saved new best (EMA) with meta: val 9.8614
step: 300, lr 2.508333e-05, train 8.676 (bpc 12.517)  val 8.668 (bpc 12.505)  (diff +0.008)
✅ Saved new best (EMA) with meta: val 8.6680
step: 450, lr 3.758333e-05, train 6.808 (bpc 9.822)  val 6.805 (bpc 9.818)  (diff +0.002)
✅ Saved new best (EMA) with meta: val 6.8053
step: 600, lr 5.008333e-05, train 4.844 (bpc 6.989)  val 4.837 (bpc 6.978)  (diff +0.008)
✅ Saved new best (EMA) with meta: val 4.8366
step: 750, lr 6.258333e-05, train 3.473 (bpc 5.010)  val 3.467 (bpc 5.001)  (diff +0.006)
✅ Saved new best (EMA) with meta: val 3.4666
step: 900, lr 7.508333e-05, train 2.996 (bpc 4.323)  val 2.886 (bpc 4.163)  (diff +0.111)
✅ Saved new best (EMA) with meta: val 2.8858
step: 1050, lr 8.758333e-05, train 2.691 (bpc 3.

TypeError: 'int' object is not callable

In [ ]:
import os, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import builtins  # <-- to avoid shadowing the built-in iter()

# --------------------
# Device, seeds & hyperparams (yours)
# --------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)

# Reproducibility & perf hints
torch.manual_seed(1337); np.random.seed(1337)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass

batch_size    = 12
block_size    = 1024   # ↑ longer context; reduce batch if you hit OOM
max_iters     = 400_000
learning_rate = 2.5e-4
eval_iters    = 150           # faster evals (tune 100–200 as you like)
n_embd        = 640
n_head        = 10
n_layer       = 12
dropout       = 0.05          # friendlier regularization

CKPT_PATH = 'drive/MyDrive/txt data/models/model-09.pt'
FINAL_CKPT_PATH = 'drive/MyDrive/txt data/models/model-0-final.pt'
TEXT_PATH = "drive/MyDrive/txt data/ComData/all_text_files_version_2.txt"

PATIENCE     = 1000              # patience-only early stopping
WEIGHT_DECAY = 0.01           # AdamW weight decay (common GPT default)
GRAD_CLIP    = 1.0            # gradient clipping (norm)
EMA_DECAY    = 0.999          # EMA decay
WARMUP_STEPS = min(3_000, max(100, max_iters // 10))  # cosine warmup

os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
os.makedirs(os.path.dirname(FINAL_CKPT_PATH), exist_ok=True)

# --------------------
# Streaming buffers (tiny vs your RAM)
# --------------------
RING_TOKENS_CAP = 200_000_000   # ~800 MB as int32 (you have 80 GB RAM)
VAL_TOKENS_CAP  = 2_000_000     # ~16 MB as int64
WARMUP_TOKENS   = 2 * batch_size * (block_size + 1)

# --------------------
# Stream file (no giant strings)
# --------------------
def stream_text_chunks(path, chunk_size=1<<20):  # ~1 MB
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        while True:
            s = f.read(chunk_size)
            if not s:
                break
            yield s

# --------------------
# Load tokenizer meta FIRST (if exists), else build from text
# --------------------
string_to_int, int_to_string, vocab_size = None, None, None

meta_ckpt = None
try:
    meta_ckpt = torch.load(CKPT_PATH, map_location='cpu')
except FileNotFoundError:
    meta_ckpt = None

if meta_ckpt is not None and 'meta' in meta_ckpt:
    meta = meta_ckpt['meta']
    string_to_int = meta['stoi']; int_to_string = meta['itos']
    vocab_size = meta['vocab_size']
    def encode(s):  return [string_to_int[c] for c in s if c in string_to_int]
    def decode(ids): return ''.join(int_to_string[i] for i in ids)
    print(f"Loaded tokenizer from {CKPT_PATH} (vocab_size={vocab_size})")
else:
    # Build from the text
    charset = set()
    for chunk in stream_text_chunks(TEXT_PATH):
        charset.update(chunk)
    charset.add("\n")
    chars = sorted(list(charset))
    vocab_size = len(chars)
    string_to_int = {ch: i for i, ch in enumerate(chars)}
    int_to_string = {i: ch for i, ch in enumerate(chars)}
    def encode(s):  return [string_to_int[c] for c in s if c in string_to_int]
    def decode(ids): return ''.join(int_to_string[i] for i in ids)
    print(f"Built tokenizer from text (vocab_size={vocab_size})")

print(f"vocab_size: {vocab_size}")

# --------------------
# Ring buffer for training tokens
# --------------------
class TokenRing:
    def __init__(self, capacity_tokens):
        self.capacity = int(capacity_tokens)
        self.buf = np.zeros(self.capacity, dtype=np.int32)
        self.write = 0
        self.filled = 0

    def add_many(self, token_list):
        """Robust, wrap-safe append; trims if chunk > capacity."""
        t = np.asarray(token_list, dtype=np.int32)
        n = int(t.shape[0])
        if n == 0:
            return
        if n >= self.capacity:
            self.buf[:] = t[-self.capacity:]
            self.write = 0
            self.filled = self.capacity
            return
        a = self.write
        end = a + n
        if end <= self.capacity:
            self.buf[a:end] = t
        else:
            first = self.capacity - a
            self.buf[a:] = t[:first]
            self.buf[:end - self.capacity] = t[first:]
        self.write = end % self.capacity
        self.filled = min(self.capacity, self.filled + n)

    def available(self):
        return self.filled

    def sample_batch(self, B, T):
        if self.filled < T + 1:
            raise RuntimeError("Ring underflow; not enough tokens.")
        if self.filled < self.capacity:
            hi = self.filled - (T + 1)
            starts = np.random.randint(0, hi + 1, size=B, dtype=np.int64)
            x = np.stack([self.buf[s:s+T] for s in starts], axis=0)
            y = np.stack([self.buf[s+1:s+T+1] for s in starts], axis=0)
        else:
            w = self.write
            seg1_len = max(0, w - (T + 1))
            seg2_len = max(0, (self.capacity - (T + 1)) - w)
            if seg1_len + seg2_len <= 0:
                raise RuntimeError("No valid sample positions.")
            p1 = seg1_len / (seg1_len + seg2_len) if (seg1_len + seg2_len) > 0 else 0.0
            n1 = np.random.binomial(B, p1)
            n2 = B - n1
            s1 = np.random.randint(0, seg1_len + 1, size=n1, dtype=np.int64) if n1>0 else np.empty(0, np.int64)
            s2 = np.random.randint(w, w + seg2_len + 1, size=n2, dtype=np.int64) if n2>0 else np.empty(0, np.int64)
            starts = np.concatenate([s1, s2], axis=0)

            def take(start):
                end = start + T + 1
                if end <= self.capacity:
                    return self.buf[start:end]
                k = end - self.capacity
                return np.concatenate([self.buf[start:], self.buf[:k]], axis=0)

            windows = np.stack([take(s) for s in starts], axis=0)
            x = windows[:, :-1]; y = windows[:, 1:]

        x = torch.from_numpy(x.astype(np.int64)).to(device)
        y = torch.from_numpy(y.astype(np.int64)).to(device)
        return x, y

ring = TokenRing(RING_TOKENS_CAP)

# --------------------
# Build val buffer + prefill ring to CAPACITY, then keep an iterator for top-ups
# --------------------
val_tokens = []

# First pass: fill validation up to cap; overflow into ring
for chunk in stream_text_chunks(TEXT_PATH):
    ids = encode(chunk + "\n")
    if not ids:
        continue
    if len(val_tokens) < VAL_TOKENS_CAP:
        need = VAL_TOKENS_CAP - len(val_tokens)
        take = min(need, len(ids))
        val_tokens.extend(ids[:take])
        leftover = ids[take:]
        if leftover:
            ring.add_many(leftover)
    else:
        ring.add_many(ids)

# Iterator for subsequent refills (use builtins.iter explicitly)
_chunk_iter = builtins.iter(stream_text_chunks(TEXT_PATH))

def refill_ring_to(target_tokens):
    """Keep reading stream until ring.available() >= target_tokens (wraps file if needed)."""
    global _chunk_iter
    while ring.available() < target_tokens:
        try:
            s = next(_chunk_iter)
        except StopIteration:
            _chunk_iter = builtins.iter(stream_text_chunks(TEXT_PATH))
            s = next(_chunk_iter)
        ring.add_many(encode(s + "\n"))

# Ensure we can sample, then fill the ring all the way
if ring.available() < WARMUP_TOKENS:
    refill_ring_to(WARMUP_TOKENS)
refill_ring_to(RING_TOKENS_CAP)

# Finalize validation buffer (DO NOT shuffle token order)
val_arr = np.array(val_tokens, dtype=np.int64); del val_tokens

print(f"val buffer tokens: {len(val_arr):,} ; ring filled: {ring.available():,}")

def get_val_batch():
    hi = len(val_arr) - (block_size + 1)
    idxs = np.random.randint(0, hi + 1, size=batch_size, dtype=np.int64)
    x = np.stack([val_arr[s:s+block_size] for s in idxs], axis=0)
    y = np.stack([val_arr[s+1:s+block_size+1] for s in idxs], axis=0)
    return torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)

# Aggressive churn across the dataset
TOKENS_PER_TOPUP = 100 * batch_size * (block_size + 1)

def top_up_ring(min_add_tokens=TOKENS_PER_TOPUP):
    global _chunk_iter
    added = 0
    while added < min_add_tokens:
        try:
            s = next(_chunk_iter)
        except StopIteration:
            _chunk_iter = builtins.iter(stream_text_chunks(TEXT_PATH))
            s = next(_chunk_iter)
        ids = encode(s + "\n")
        if not ids:
            continue
        ring.add_many(ids)
        added += len(ids)

# --------------------
# Model (GPT-like)
# --------------------
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x); q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1); wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj  = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd_):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd_, 4 * n_embd_),
            nn.ReLU(),
            nn.Linear(4 * n_embd_, n_embd_),
            nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd_, n_head_):
        super().__init__()
        head_size = n_embd_ // n_head_
        self.sa = MultiHeadAttention(n_head_, head_size)
        self.ffwd = FeedForward(n_embd_)
        self.ln1 = nn.LayerNorm(n_embd_)
        self.ln2 = nn.LayerNorm(n_embd_)
    def forward(self, x):
        y = self.sa(x); x = self.ln1(x + y)
        y = self.ffwd(x); x = self.ln2(x + y)
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size_):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table    = nn.Embedding(vocab_size_, n_embd)
        self.position_embedding_table = nn.Embedding(self.block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size_, bias=False)
        # Init
        self.apply(self._init_weights)
        # Tie weights
        self.lm_head.weight = self.token_embedding_table.weight

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if getattr(m, 'bias', None) is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        if index.size(1) > self.block_size:
            index = index[:, -self.block_size:]
        B, T = index.shape
        tok = self.token_embedding_table(index)
        pos = torch.arange(T, device=index.device)
        pos_emb = self.position_embedding_table(pos)
        x = tok + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            if targets.size(1) > self.block_size:
                targets = targets[:, -self.block_size:]
            B2, T2, C = logits.shape
            loss = F.cross_entropy(logits.view(B2*T2, C), targets.view(B2*T2))
        return logits, loss

    @torch.no_grad()
    def generate(self, index, max_new_tokens):
        self.eval()
        idx = index
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            nxt = torch.multinomial(probs, 1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

# --------------------
# Meta + saving
# --------------------
def build_meta():
    return {
        'stoi': string_to_int,
        'itos': int_to_string,
        'vocab_size': len(string_to_int),
        'block_size': block_size,
        'n_embd': n_embd, 'n_head': n_head, 'n_layer': n_layer,
    }

def atomic_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

def save_ckpt(path, iter_idx, best_val, model_state, ema_state, optimizer_state, scheduler_state):
    atomic_save({
        'model_state': model_state,
        'ema_state': ema_state,
        'optimizer_state': optimizer_state,
        'scheduler_state': scheduler_state,
        'iter': iter_idx,
        'best_val_loss': best_val,
        'meta': build_meta(),
    }, path)

# --------------------
# Init model/EMA/opt, optional resume (weights later)
# --------------------
model = GPTLanguageModel(vocab_size).to(device)
ema_model = GPTLanguageModel(vocab_size).to(device)
ema_model.load_state_dict(model.state_dict())
for p in ema_model.parameters():
    p.requires_grad_(False)

# Weight decay: exclude biases & LayerNorm from WD
decay, no_decay, seen = [], [], set()
for n, p in model.named_parameters():
    if id(p) in seen:
        continue
    seen.add(id(p))
    if p.ndim == 1 or n.endswith('.bias'):
        no_decay.append(p)
    else:
        decay.append(p)

optimizer = torch.optim.AdamW(
    [{'params': decay, 'weight_decay': WEIGHT_DECAY},
     {'params': no_decay, 'weight_decay': 0.0}],
    lr=learning_rate
)

# --- AMP setup (CUDA only) ---
use_amp = (device == 'cuda')
amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# LR schedule (warmup + cosine), respecting call order (scheduler.step AFTER optimizer.step)
def lr_lambda(step):
    step = step + 1  # because we call scheduler.step() after optimizer.step()
    if step <= WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    p = (step - WARMUP_STEPS) / max(1, max_iters - WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * p))

start_iter = 0
best_val_loss = float('inf')

# If full ckpt exists, resume
ckpt = None
try:
    ckpt = torch.load(CKPT_PATH, map_location=device)
except FileNotFoundError:
    ckpt = None

if ckpt is not None:
    try:
        model.load_state_dict(ckpt['model_state'])
        if 'ema_state' in ckpt:
            ema_model.load_state_dict(ckpt['ema_state'])
        else:
            ema_model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        start_iter = ckpt.get('iter', 0)
        best_val_loss = ckpt.get('best_val_loss', float('inf'))
        print(f"Resumed from {CKPT_PATH} @ iter={start_iter}, best_val={best_val_loss:.4f}")
    except Exception as e:
        print(f"Warning: failed to load weights/optimizer from checkpoint: {e}")
        ema_model.load_state_dict(model.state_dict())

# Scheduler created after knowing start_iter, then optionally load state
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda, last_epoch=start_iter - 1
)
if ckpt is not None and 'scheduler_state' in ckpt:
    try:
        scheduler.load_state_dict(ckpt['scheduler_state'])
    except Exception as e:
        print(f"Warning: failed to load scheduler state: {e}")

# Optional: compile for speed (PyTorch 2.x)
# We will handle the state dict prefix manually during load
try:
    model = torch.compile(model)
    # EMA model should not be compiled for state dict consistency
    # ema_model = torch.compile(ema_model)
except Exception:
    pass

# --------------------
# Eval (uses EMA)
# --------------------
@torch.no_grad()
def estimate_loss(eval_model):
    prev_mode = eval_model.training
    eval_model.eval()
    try:
        out = {}
        # train loss from ring
        tl = []
        for _ in range(eval_iters):
            X, Y = ring.sample_batch(batch_size, block_size)
            with torch.autocast(device_type='cuda', dtype=amp_dtype, enabled=use_amp):
                _, loss = eval_model(X, Y)
            tl.append(loss.item())
        out['train'] = float(np.mean(tl))
        # val loss from fixed buffer
        vl = []
        for _ in range(eval_iters):
            X, Y = get_val_batch()
            with torch.autocast(device_type='cuda', dtype=amp_dtype, enabled=use_amp):
                _, loss = eval_model(X, Y)
            vl.append(loss.item())
        out['val'] = float(np.mean(vl))
        return out
    finally:
        eval_model.train(prev_mode)

def ema_update():
    with torch.no_grad():
        msd = model.state_dict()
        esd = ema_model.state_dict()
        for k in msd.keys():
            # Handle potential _orig_mod prefix if the main model was compiled
            if k.startswith('_orig_mod.'):
                esd[k[len('_orig_mod.'):]].mul_(EMA_DECAY).add_(msd[k], alpha=1.0 - EMA_DECAY)
            else:
                esd[k].mul_(EMA_DECAY).add_(msd[k], alpha=1.0 - EMA_DECAY)

# --------------------
# Training (samples from ring; patience-only early stop)
# --------------------
no_improve = 0
final_loss = None
last_iter = start_iter

for step in range(start_iter, max_iters):   # <-- renamed from `for iter in ...`
    last_iter = step

    # Aggressively refresh training pool EVERY STEP
    top_up_ring()

    # Periodic eval with EMA weights
    if step % eval_iters == 0:              # <-- use step
        lr_now = scheduler.get_last_lr()[0]
        losses = estimate_loss(ema_model)
        train_loss = losses['train']; val_loss = losses['val']
        diff = train_loss - val_loss
        # also print bits-per-char (bpc)
        bpc_train = train_loss / math.log(2)
        bpc_val   = val_loss   / math.log(2)
        print(f"step: {step}, lr {lr_now:.6e}, "
              f"train {train_loss:.3f} (bpc {bpc_train:.3f})  "
              f"val {val_loss:.3f} (bpc {bpc_val:.3f})  "
              f"(diff {diff:+.3f})")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_ckpt(CKPT_PATH, step, best_val_loss,   # <-- pass step
                      model.state_dict(), ema_model.state_dict(),
                      optimizer.state_dict(), scheduler.state_dict())
            no_improve = 0
            print(f"✅ Saved new best (EMA) with meta: val {best_val_loss:.4f}")
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"⏹️ Early stop: no validation improvement for {PATIENCE} evals.")
            break

    # Train step (AMP)
    optimizer.zero_grad(set_to_none=True)
    xb, yb = ring.sample_batch(batch_size, block_size)
    with torch.autocast(device_type='cuda', dtype=amp_dtype, enabled=use_amp):
        logits, loss = model(xb, yb)

    scaler.scale(loss).backward()
    # Important: unscale before gradient clipping
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

    scaler.step(optimizer)
    scaler.update()

    scheduler.step()
    ema_update()
    final_loss = float(loss.item())

print("done. last minibatch loss:", None if final_loss is None else f"{final_loss:.4f}")
print(f"Best validation loss observed: {best_val_loss:.4f}")
print(f"Best checkpoint saved to: {CKPT_PATH}")

save_ckpt(FINAL_CKPT_PATH, last_iter, best_val_loss,
          model.state_dict(), ema_model.state_dict(),
          optimizer.state_dict(), scheduler.state_dict())
print(f"Final checkpoint (EMA + meta) saved to: {FINAL_CKPT_PATH}")

# --------------------
# Text generation (reload best EMA)
# --------------------
with torch.no_grad():
    try:
        ckpt = torch.load(CKPT_PATH, map_location=device)
    except FileNotFoundError:
        ckpt = torch.load(FINAL_CKPT_PATH, map_location=device)

    meta = ckpt['meta']
    string_to_int = meta['stoi']; int_to_string = meta['itos']
    # Rebuild the model for generation if vocab size changed
    if model.lm_head.out_features != meta['vocab_size']:
         vocab_size = meta['vocab_size']
         gen_model = GPTLanguageModel(vocab_size).to(device)
    else:
         gen_model = GPTLanguageModel(vocab_size).to(device)

    state = ckpt.get('ema_state', ckpt['model_state'])
    cleaned_state = {k.replace('_orig_mod.', ''): v for k, v in state.items()}
    gen_model.load_state_dict(cleaned_state)
    gen_model.eval()
    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    print(decode(gen_model.generate(context, max_new_tokens=500)[0].tolist()))


device: cuda
Loaded tokenizer from drive/MyDrive/txt data/models/model-08.pt (vocab_size=31336)
vocab_size: 31336
val buffer tokens: 2,000,000 ; ring filled: 200,000,000


/tmp/ipython-input-219021115.py:402: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


	Missing key(s) in state_dict: "token_embedding_table.weight", "position_embedding_table.weight", "blocks.0.sa.heads.0.tril", "blocks.0.sa.heads.0.key.weight", "blocks.0.sa.heads.0.query.weight", "blocks.0.sa.heads.0.value.weight", "blocks.0.sa.heads.1.tril", "blocks.0.sa.heads.1.key.weight", "blocks.0.sa.heads.1.query.weight", "blocks.0.sa.heads.1.value.weight", "blocks.0.sa.heads.2.tril", "blocks.0.sa.heads.2.key.weight", "blocks.0.sa.heads.2.query.weight", "blocks.0.sa.heads.2.value.weight", "blocks.0.sa.heads.3.tril", "blocks.0.sa.heads.3.key.weight", "blocks.0.sa.heads.3.query.weight", "blocks.0.sa.heads.3.value.weight", "blocks.0.sa.heads.4.tril", "blocks.0.sa.heads.4.key.weight", "blocks.0.sa.heads.4.query.weight", "blocks.0.sa.heads.4.value.weight", "blocks.0.sa.heads.5.tril", "blocks.0.sa.heads.5.key.weight", "blocks.0.sa.heads.5.query.weight", "blocks.0.sa.heads.5.value.weight", "blocks.0.sa.heads.6.tril", "blocks.0.sa.heads.6.key.weight", "blocks.0.sa.heads.6.query.weight", 

In [ ]:
import os, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import builtins  # <-- to avoid shadowing the built-in iter()

# --------------------
# Device, seeds & perf
# --------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)

torch.manual_seed(1337); np.random.seed(1337)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass

# ===== Model size / context =====
block_size    = 1024
n_embd        = 640
n_head        = 10
n_layer       = 12
dropout       = 0.05

# ===== Batching (micro-batch + grad accumulation) =====
batch_size         = 8                 # per micro-step batch
GRAD_ACCUM_STEPS   = 12                # effective batch = 8 * 12 = 96 sequences/step

# ===== Train schedule =====
max_iters     = 400_000
learning_rate = 2.5e-4
eval_iters    = 150                     # also used as eval frequency (every N steps)
PATIENCE      = 100
WEIGHT_DECAY  = 0.01
GRAD_CLIP     = 1.0
EMA_DECAY     = 0.999
WARMUP_STEPS  = min(3_000, max(100, max_iters // 10))

# ===== Paths =====
CKPT_PATH       = 'drive/MyDrive/txt data/models/model-10.pt'
FINAL_CKPT_PATH = 'drive/MyDrive/txt data/models/model-10-final.pt'  # fixed
TEXT_PATH       = "drive/MyDrive/txt data/ComData/all_text_files_version_2_clean.txt"

os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
os.makedirs(os.path.dirname(FINAL_CKPT_PATH), exist_ok=True)

# --------------------
# Streaming buffers (scaled for 80 GB RAM)
# --------------------
# int32 -> ~4 bytes/token
RING_TOKENS_CAP = 500_000_000   # ~2.0 GB RAM
VAL_TOKENS_CAP  = 5_000_000     # ~40 MB
WARMUP_TOKENS   = 2 * batch_size * (block_size + 1)

# Aggressive churn across the dataset
TOKENS_PER_TOPUP = 100 * batch_size * (block_size + 1) * GRAD_ACCUM_STEPS

# --------------------
# Stream file (no giant strings)
# --------------------
def stream_text_chunks(path, chunk_size=1<<20):  # ~1 MB
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        while True:
            s = f.read(chunk_size)
            if not s:
                break
            yield s

# --------------------
# Load tokenizer meta FIRST (if exists), else build from text
# --------------------
string_to_int, int_to_string, vocab_size = None, None, None

meta_ckpt = None
try:
    meta_ckpt = torch.load(CKPT_PATH, map_location='cpu')
except FileNotFoundError:
    meta_ckpt = None

if meta_ckpt is not None and 'meta' in meta_ckpt:
    meta = meta_ckpt['meta']
    string_to_int = meta['stoi']; int_to_string = meta['itos']
    vocab_size = meta['vocab_size']
    def encode(s):  return [string_to_int[c] for c in s if c in string_to_int]
    def decode(ids): return ''.join(int_to_string[i] for i in ids)
    print(f"Loaded tokenizer from {CKPT_PATH} (vocab_size={vocab_size})")
else:
    # Build from the text
    charset = set()
    for chunk in stream_text_chunks(TEXT_PATH):
        charset.update(chunk)
    charset.add("\n")
    chars = sorted(list(charset))
    vocab_size = len(chars)
    string_to_int = {ch: i for i, ch in enumerate(chars)}
    int_to_string = {i: ch for i, ch in enumerate(chars)}
    def encode(s):  return [string_to_int[c] for c in s if c in string_to_int]
    def decode(ids): return ''.join(int_to_string[i] for i in ids)
    print(f"Built tokenizer from text (vocab_size={vocab_size})")

print(f"vocab_size: {vocab_size}")

# --------------------
# Ring buffer for training tokens
# --------------------
class TokenRing:
    def __init__(self, capacity_tokens):
        self.capacity = int(capacity_tokens)
        self.buf = np.zeros(self.capacity, dtype=np.int32)
        self.write = 0
        self.filled = 0

    def add_many(self, token_list):
        """Robust, wrap-safe append; trims if chunk > capacity."""
        t = np.asarray(token_list, dtype=np.int32)
        n = int(t.shape[0])
        if n == 0:
            return
        if n >= self.capacity:
            self.buf[:] = t[-self.capacity:]
            self.write = 0
            self.filled = self.capacity
            return
        a = self.write
        end = a + n
        if end <= self.capacity:
            self.buf[a:end] = t
        else:
            first = self.capacity - a
            self.buf[a:] = t[:first]
            self.buf[:end - self.capacity] = t[first:]
        self.write = end % self.capacity
        self.filled = min(self.capacity, self.filled + n)

    def available(self):
        return self.filled

    def sample_batch(self, B, T):
        if self.filled < T + 1:
            raise RuntimeError("Ring underflow; not enough tokens.")
        if self.filled < self.capacity:
            hi = self.filled - (T + 1)
            starts = np.random.randint(0, hi + 1, size=B, dtype=np.int64)
            x = np.stack([self.buf[s:s+T] for s in starts], axis=0)
            y = np.stack([self.buf[s+1:s+T+1] for s in starts], axis=0)
        else:
            w = self.write
            seg1_len = max(0, w - (T + 1))
            seg2_len = max(0, (self.capacity - (T + 1)) - w)
            if seg1_len + seg2_len <= 0:
                raise RuntimeError("No valid sample positions.")
            p1 = seg1_len / (seg1_len + seg2_len) if (seg1_len + seg2_len) > 0 else 0.0
            n1 = np.random.binomial(B, p1)
            n2 = B - n1
            s1 = np.random.randint(0, seg1_len + 1, size=n1, dtype=np.int64) if n1>0 else np.empty(0, np.int64)
            s2 = np.random.randint(w, w + seg2_len + 1, size=n2, dtype=np.int64) if n2>0 else np.empty(0, np.int64)
            starts = np.concatenate([s1, s2], axis=0)

            def take(start):
                end = start + T + 1
                if end <= self.capacity:
                    return self.buf[start:end]
                k = end - self.capacity
                return np.concatenate([self.buf[start:], self.buf[:k]], axis=0)

            windows = np.stack([take(s) for s in starts], axis=0)
            x = windows[:, :-1]; y = windows[:, 1:]

        x = torch.from_numpy(x.astype(np.int64)).to(device)
        y = torch.from_numpy(y.astype(np.int64)).to(device)
        return x, y

ring = TokenRing(RING_TOKENS_CAP)

# --------------------
# Build val buffer + prefill ring to CAPACITY, then keep an iterator for top-ups
# --------------------
val_tokens = []

# First pass: fill validation up to cap; overflow into ring
for chunk in stream_text_chunks(TEXT_PATH):
    ids = encode(chunk + "\n")
    if not ids:
        continue
    if len(val_tokens) < VAL_TOKENS_CAP:
        need = VAL_TOKENS_CAP - len(val_tokens)
        take = min(need, len(ids))
        val_tokens.extend(ids[:take])
        leftover = ids[take:]
        if leftover:
            ring.add_many(leftover)
    else:
        ring.add_many(ids)

# Iterator for subsequent refills (use builtins.iter explicitly)
_chunk_iter = builtins.iter(stream_text_chunks(TEXT_PATH))

def refill_ring_to(target_tokens):
    """Keep reading stream until ring.available() >= target_tokens (wraps file if needed)."""
    global _chunk_iter
    while ring.available() < target_tokens:
        try:
            s = next(_chunk_iter)
        except StopIteration:
            _chunk_iter = builtins.iter(stream_text_chunks(TEXT_PATH))
            s = next(_chunk_iter)
        ring.add_many(encode(s + "\n"))

# Ensure we can sample, then fill the ring all the way
if ring.available() < WARMUP_TOKENS:
    refill_ring_to(WARMUP_TOKENS)
refill_ring_to(RING_TOKENS_CAP)

# Finalize validation buffer (DO NOT shuffle token order)
val_arr = np.array(val_tokens, dtype=np.int64); del val_tokens

print(f"val buffer tokens: {len(val_arr):,} ; ring filled: {ring.available():,}")

def get_val_batch():
    hi = len(val_arr) - (block_size + 1)
    idxs = np.random.randint(0, hi + 1, size=batch_size, dtype=np.int64)
    x = np.stack([val_arr[s:s+block_size] for s in idxs], axis=0)
    y = np.stack([val_arr[s+1:s+block_size+1] for s in idxs], axis=0)
    return torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)

def top_up_ring(min_add_tokens=TOKENS_PER_TOPUP):
    """Aggressively add tokens to keep training pool fresh."""
    global _chunk_iter
    added = 0
    while added < min_add_tokens:
        try:
            s = next(_chunk_iter)
        except StopIteration:
            _chunk_iter = builtins.iter(stream_text_chunks(TEXT_PATH))
            s = next(_chunk_iter)
        ids = encode(s + "\n")
        if not ids:
            continue
        ring.add_many(ids)
        added += len(ids)

# --------------------
# Model (GPT-like)
# --------------------
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x); q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1); wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj  = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd_):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd_, 4 * n_embd_),
            nn.ReLU(),
            nn.Linear(4 * n_embd_, n_embd_),
            nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd_, n_head_):
        super().__init__()
        head_size = n_embd_ // n_head_
        self.sa = MultiHeadAttention(n_head_, head_size)
        self.ffwd = FeedForward(n_embd_)
        self.ln1 = nn.LayerNorm(n_embd_)
        self.ln2 = nn.LayerNorm(n_embd_)
    def forward(self, x):
        y = self.sa(x); x = self.ln1(x + y)
        y = self.ffwd(x); x = self.ln2(x + y)
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size_):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table    = nn.Embedding(vocab_size_, n_embd)
        self.position_embedding_table = nn.Embedding(self.block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size_, bias=False)
        # Init
        self.apply(self._init_weights)
        # Tie weights
        self.lm_head.weight = self.token_embedding_table.weight

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if getattr(m, 'bias', None) is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        if index.size(1) > self.block_size:
            index = index[:, -self.block_size:]
        B, T = index.shape
        tok = self.token_embedding_table(index)
        pos = torch.arange(T, device=index.device)
        pos_emb = self.position_embedding_table(pos)
        x = tok + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            if targets.size(1) > self.block_size:
                targets = targets[:, -self.block_size:]
            B2, T2, C = logits.shape
            loss = F.cross_entropy(logits.view(B2*T2, C), targets.view(B2*T2))
        return logits, loss

    @torch.no_grad()
    def generate(self, index, max_new_tokens):
        self.eval()
        idx = index
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            nxt = torch.multinomial(probs, 1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

# --------------------
# Meta + saving
# --------------------
def build_meta():
    return {
        'stoi': string_to_int,
        'itos': int_to_string,
        'vocab_size': len(string_to_int),
        'block_size': block_size,
        'n_embd': n_embd, 'n_head': n_head, 'n_layer': n_layer,
    }

def atomic_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

def save_ckpt(path, iter_idx, best_val, model_state, ema_state, optimizer_state, scheduler_state):
    atomic_save({
        'model_state': model_state,
        'ema_state': ema_state,
        'optimizer_state': optimizer_state,
        'scheduler_state': scheduler_state,
        'iter': iter_idx,
        'best_val_loss': best_val,
        'meta': build_meta(),
    }, path)

# --------------------
# Init model/EMA/opt, optional resume (weights later)
# --------------------
model = GPTLanguageModel(vocab_size).to(device)
ema_model = GPTLanguageModel(vocab_size).to(device)
ema_model.load_state_dict(model.state_dict())
for p in ema_model.parameters():
    p.requires_grad_(False)

# Weight decay: exclude biases & LayerNorm from WD
decay, no_decay, seen = [], [], set()
for n, p in model.named_parameters():
    if id(p) in seen:
        continue
    seen.add(id(p))
    if p.ndim == 1 or n.endswith('.bias'):
        no_decay.append(p)
    else:
        decay.append(p)

optimizer = torch.optim.AdamW(
    [{'params': decay, 'weight_decay': WEIGHT_DECAY},
     {'params': no_decay, 'weight_decay': 0.0}],
    lr=learning_rate
)

# --- AMP setup (CUDA only) ---
use_amp = (device == 'cuda')
amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
# GradScaler is only useful for fp16 (not bf16)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp and amp_dtype is torch.float16)

# LR schedule (warmup + cosine), respecting call order (scheduler.step AFTER optimizer.step)
def lr_lambda(step):
    step = step + 1  # because we call scheduler.step() after optimizer.step()
    if step <= WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    p = (step - WARMUP_STEPS) / max(1, max_iters - WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * p))

start_iter = 0
best_val_loss = float('inf')

# If full ckpt exists, resume
ckpt = None
try:
    ckpt = torch.load(CKPT_PATH, map_location=device)
except FileNotFoundError:
    ckpt = None

if ckpt is not None:
    try:
        model.load_state_dict(ckpt['model_state'])
        if 'ema_state' in ckpt:
            ema_model.load_state_dict(ckpt['ema_state'])
        else:
            ema_model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        start_iter = ckpt.get('iter', 0)
        best_val_loss = ckpt.get('best_val_loss', float('inf'))
        print(f"Resumed from {CKPT_PATH} @ iter={start_iter}, best_val={best_val_loss:.4f}")
    except Exception as e:
        print(f"Warning: failed to load weights/optimizer from checkpoint: {e}")
        ema_model.load_state_dict(model.state_dict())

# Scheduler created after knowing start_iter, then optionally load state
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda, last_epoch=start_iter - 1
)
if ckpt is not None and 'scheduler_state' in ckpt:
    try:
        scheduler.load_state_dict(ckpt['scheduler_state'])
    except Exception as e:
        print(f"Warning: failed to load scheduler state: {e}")

# Optional: compile for speed (PyTorch 2.x)
try:
    model = torch.compile(model)
except Exception:
    pass

# --------------------
# Eval (uses EMA)
# --------------------
@torch.no_grad()
def estimate_loss(eval_model):
    prev_mode = eval_model.training
    eval_model.eval()
    try:
        out = {}
        # train loss from ring
        tl = []
        for _ in range(eval_iters):
            X, Y = ring.sample_batch(batch_size, block_size)
            with torch.autocast(device_type='cuda', dtype=amp_dtype, enabled=use_amp):
                _, loss = eval_model(X, Y)
            tl.append(loss.item())
        out['train'] = float(np.mean(tl))
        # val loss from fixed buffer
        vl = []
        for _ in range(eval_iters):
            X, Y = get_val_batch()
            with torch.autocast(device_type='cuda', dtype=amp_dtype, enabled=use_amp):
                _, loss = eval_model(X, Y)
            vl.append(loss.item())
        out['val'] = float(np.mean(vl))
        return out
    finally:
        eval_model.train(prev_mode)

def ema_update():
    with torch.no_grad():
        msd = model.state_dict()
        esd = ema_model.state_dict()
        for k in msd.keys():
            # Handle potential _orig_mod prefix if the main model was compiled
            if k.startswith('_orig_mod.'):
                esd[k[len('_orig_mod.'):]].mul_(EMA_DECAY).add_(msd[k], alpha=1.0 - EMA_DECAY)
            else:
                esd[k].mul_(EMA_DECAY).add_(msd[k], alpha=1.0 - EMA_DECAY)

# --------------------
# Training (grad accumulation + AMP; patience-only early stop)
# --------------------
no_improve = 0
final_loss = None
last_iter = start_iter

for step in range(start_iter, max_iters):
    last_iter = step

    # Keep training pool fresh
    top_up_ring()

    # Periodic eval with EMA weights
    if step % eval_iters == 0:
        lr_now = scheduler.get_last_lr()[0]
        losses = estimate_loss(ema_model)
        train_loss = losses['train']; val_loss = losses['val']
        diff = train_loss - val_loss
        bpc_train = train_loss / math.log(2)
        bpc_val   = val_loss   / math.log(2)
        print(f"step: {step}, lr {lr_now:.6e}, "
              f"train {train_loss:.3f} (bpc {bpc_train:.3f})  "
              f"val {val_loss:.3f} (bpc {bpc_val:.3f})  "
              f"(diff {diff:+.3f})")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_ckpt(CKPT_PATH, step, best_val_loss,
                      model.state_dict(), ema_model.state_dict(),
                      optimizer.state_dict(), scheduler.state_dict())
            no_improve = 0
            print(f"✅ Saved new best (EMA) with meta: val {best_val_loss:.4f}")
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"⏹️ Early stop: no validation improvement for {PATIENCE} evals.")
            break

    # ---- Train step with gradient accumulation ----
    optimizer.zero_grad(set_to_none=True)
    loss_accum = 0.0

    for _ in range(GRAD_ACCUM_STEPS):
        xb, yb = ring.sample_batch(batch_size, block_size)
        with torch.autocast(device_type='cuda', dtype=amp_dtype, enabled=use_amp):
            _, loss = model(xb, yb)
            loss = loss / GRAD_ACCUM_STEPS  # normalize per micro-step

        if scaler is not None and scaler.is_enabled():
            scaler.scale(loss).backward()
        else:
            loss.backward()

        loss_accum += float(loss.item())

    # Unscale (if needed), clip, step
    if scaler is not None and scaler.is_enabled():
        scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

    if scaler is not None and scaler.is_enabled():
        scaler.step(optimizer)
        scaler.update()
    else:
        optimizer.step()

    scheduler.step()
    ema_update()
    final_loss = loss_accum

print("done. last minibatch (accumulated) loss:", None if final_loss is None else f"{final_loss:.4f}")
print(f"Best validation loss observed: {best_val_loss:.4f}")
print(f"Best checkpoint saved to: {CKPT_PATH}")

save_ckpt(FINAL_CKPT_PATH, last_iter, best_val_loss,
          model.state_dict(), ema_model.state_dict(),
          optimizer.state_dict(), scheduler.state_dict())
print(f"Final checkpoint (EMA + meta) saved to: {FINAL_CKPT_PATH}")

# --------------------
# Text generation (reload best EMA)
# --------------------
with torch.no_grad():
    try:
        ckpt = torch.load(CKPT_PATH, map_location=device)
    except FileNotFoundError:
        ckpt = torch.load(FINAL_CKPT_PATH, map_location=device)

    meta = ckpt['meta']
    string_to_int = meta['stoi']; int_to_string = meta['itos']

    # Rebuild the model for generation if vocab size changed
    if 'vocab_size' in meta and vocab_size != meta['vocab_size']:
        vocab_size = meta['vocab_size']
    gen_model = GPTLanguageModel(vocab_size).to(device)

    state = ckpt.get('ema_state', ckpt['model_state'])
    cleaned_state = {k.replace('_orig_mod.', ''): v for k, v in state.items()}
    gen_model.load_state_dict(cleaned_state)
    gen_model.eval()
    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    print(decode(gen_model.generate(context, max_new_tokens=500)[0].tolist()))


device: cuda
Loaded tokenizer from drive/MyDrive/txt data/models/model-10.pt (vocab_size=99)
vocab_size: 99
val buffer tokens: 5,000,000 ; ring filled: 500,000,000


/tmp/ipython-input-89304650.py:410: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp and amp_dtype is torch.float16)


	Missing key(s) in state_dict: "token_embedding_table.weight", "position_embedding_table.weight", "blocks.0.sa.heads.0.tril", "blocks.0.sa.heads.0.key.weight", "blocks.0.sa.heads.0.query.weight", "blocks.0.sa.heads.0.value.weight", "blocks.0.sa.heads.1.tril", "blocks.0.sa.heads.1.key.weight", "blocks.0.sa.heads.1.query.weight", "blocks.0.sa.heads.1.value.weight", "blocks.0.sa.heads.2.tril", "blocks.0.sa.heads.2.key.weight", "blocks.0.sa.heads.2.query.weight", "blocks.0.sa.heads.2.value.weight", "blocks.0.sa.heads.3.tril", "blocks.0.sa.heads.3.key.weight", "blocks.0.sa.heads.3.query.weight", "blocks.0.sa.heads.3.value.weight", "blocks.0.sa.heads.4.tril", "blocks.0.sa.heads.4.key.weight", "blocks.0.sa.heads.4.query.weight", "blocks.0.sa.heads.4.value.weight", "blocks.0.sa.heads.5.tril", "blocks.0.sa.heads.5.key.weight", "blocks.0.sa.heads.5.query.weight", "blocks.0.sa.heads.5.value.weight", "blocks.0.sa.heads.6.tril", "blocks.0.sa.heads.6.key.weight", "blocks.0.sa.heads.6.query.weight", 

W0928 01:07:34.134000 2278 torch/_inductor/utils.py:1436] [0/0] Not enough SMs to use max_autotune_gemm mode


KeyboardInterrupt: 

In [ ]:
import os, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import builtins  # <-- to avoid shadowing the built-in iter()

# --------------------
# Device, seeds & perf
# --------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)

torch.manual_seed(1337); np.random.seed(1337)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass

# ===== Model size / context =====
block_size    = 1024
n_embd        = 640
n_head        = 10
n_layer       = 12
dropout       = 0.06

# ===== Batching (micro-batch + grad accumulation) =====
batch_size         = 8                 # per micro-step batch
GRAD_ACCUM_STEPS   = 12                # effective batch = 8 * 12 = 96 sequences/step

# ===== Train schedule =====
max_iters     = 400_000
learning_rate = 2.5e-4
eval_iters    = 150                     # also used as eval frequency (every N steps)
PATIENCE      = 150
WEIGHT_DECAY  = 0.01
GRAD_CLIP     = 1.0
EMA_DECAY     = 0.999
WARMUP_STEPS  = min(3_000, max(100, max_iters // 10))

# ===== Paths =====
CKPT_PATH       = 'drive/MyDrive/txt data/models/plaintext-wikipedia.pt'
FINAL_CKPT_PATH = 'drive/MyDrive/txt data/models/plaintext-wikipedia-final.pt'  # fixed
TEXT_PATH       = "drive/MyDrive/txt data/Data Set 3/Plaintext Wikipedia.txt"

os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
os.makedirs(os.path.dirname(FINAL_CKPT_PATH), exist_ok=True)

# --------------------
# Streaming buffers (scaled for 80 GB RAM)
# --------------------
# int32 -> ~4 bytes/token
RING_TOKENS_CAP = 500_000_000   # ~2.0 GB RAM
VAL_TOKENS_CAP  = 5_000_000     # ~40 MB
WARMUP_TOKENS   = 2 * batch_size * (block_size + 1)

# Aggressive churn across the dataset
TOKENS_PER_TOPUP = 100 * batch_size * (block_size + 1) * GRAD_ACCUM_STEPS

# --------------------
# Stream file (no giant strings)
# --------------------
def stream_text_chunks(path, chunk_size=1<<20):  # ~1 MB
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        while True:
            s = f.read(chunk_size)
            if not s:
                break
            yield s

# --------------------
# Load tokenizer meta FIRST (if exists), else build from text
# --------------------
string_to_int, int_to_string, vocab_size = None, None, None

meta_ckpt = None
try:
    meta_ckpt = torch.load(CKPT_PATH, map_location='cpu')
except FileNotFoundError:
    meta_ckpt = None

if meta_ckpt is not None and 'meta' in meta_ckpt:
    meta = meta_ckpt['meta']
    string_to_int = meta['stoi']; int_to_string = meta['itos']
    vocab_size = meta['vocab_size']
    def encode(s):  return [string_to_int[c] for c in s if c in string_to_int]
    def decode(ids): return ''.join(int_to_string[i] for i in ids)
    print(f"Loaded tokenizer from {CKPT_PATH} (vocab_size={vocab_size})")
else:
    # Build from the text
    charset = set()
    for chunk in stream_text_chunks(TEXT_PATH):
        charset.update(chunk)
    charset.add("\n")
    chars = sorted(list(charset))
    vocab_size = len(chars)
    string_to_int = {ch: i for i, ch in enumerate(chars)}
    int_to_string = {i: ch for i, ch in enumerate(chars)}
    def encode(s):  return [string_to_int[c] for c in s if c in string_to_int]
    def decode(ids): return ''.join(int_to_string[i] for i in ids)
    print(f"Built tokenizer from text (vocab_size={vocab_size})")

print(f"vocab_size: {vocab_size}")

# --------------------
# Ring buffer for training tokens
# --------------------
class TokenRing:
    def __init__(self, capacity_tokens):
        self.capacity = int(capacity_tokens)
        self.buf = np.zeros(self.capacity, dtype=np.int32)
        self.write = 0
        self.filled = 0

    def add_many(self, token_list):
        """Robust, wrap-safe append; trims if chunk > capacity."""
        t = np.asarray(token_list, dtype=np.int32)
        n = int(t.shape[0])
        if n == 0:
            return
        if n >= self.capacity:
            self.buf[:] = t[-self.capacity:]
            self.write = 0
            self.filled = self.capacity
            return
        a = self.write
        end = a + n
        if end <= self.capacity:
            self.buf[a:end] = t
        else:
            first = self.capacity - a
            self.buf[a:] = t[:first]
            self.buf[:end - self.capacity] = t[first:]
        self.write = end % self.capacity
        self.filled = min(self.capacity, self.filled + n)

    def available(self):
        return self.filled

    def sample_batch(self, B, T):
        if self.filled < T + 1:
            raise RuntimeError("Ring underflow; not enough tokens.")
        if self.filled < self.capacity:
            hi = self.filled - (T + 1)
            starts = np.random.randint(0, hi + 1, size=B, dtype=np.int64)
            x = np.stack([self.buf[s:s+T] for s in starts], axis=0)
            y = np.stack([self.buf[s+1:s+T+1] for s in starts], axis=0)
        else:
            w = self.write
            seg1_len = max(0, w - (T + 1))
            seg2_len = max(0, (self.capacity - (T + 1)) - w)
            if seg1_len + seg2_len <= 0:
                raise RuntimeError("No valid sample positions.")
            p1 = seg1_len / (seg1_len + seg2_len) if (seg1_len + seg2_len) > 0 else 0.0
            n1 = np.random.binomial(B, p1)
            n2 = B - n1
            s1 = np.random.randint(0, seg1_len + 1, size=n1, dtype=np.int64) if n1>0 else np.empty(0, np.int64)
            s2 = np.random.randint(w, w + seg2_len + 1, size=n2, dtype=np.int64) if n2>0 else np.empty(0, np.int64)
            starts = np.concatenate([s1, s2], axis=0)

            def take(start):
                end = start + T + 1
                if end <= self.capacity:
                    return self.buf[start:end]
                k = end - self.capacity
                return np.concatenate([self.buf[start:], self.buf[:k]], axis=0)

            windows = np.stack([take(s) for s in starts], axis=0)
            x = windows[:, :-1]; y = windows[:, 1:]

        x = torch.from_numpy(x.astype(np.int64)).to(device)
        y = torch.from_numpy(y.astype(np.int64)).to(device)
        return x, y

ring = TokenRing(RING_TOKENS_CAP)

# --------------------
# Build val buffer + prefill ring to CAPACITY, then keep an iterator for top-ups
# --------------------
val_tokens = []

# First pass: fill validation up to cap; overflow into ring
for chunk in stream_text_chunks(TEXT_PATH):
    ids = encode(chunk + "\n")
    if not ids:
        continue
    if len(val_tokens) < VAL_TOKENS_CAP:
        need = VAL_TOKENS_CAP - len(val_tokens)
        take = min(need, len(ids))
        val_tokens.extend(ids[:take])
        leftover = ids[take:]
        if leftover:
            ring.add_many(leftover)
    else:
        ring.add_many(ids)

# Iterator for subsequent refills (use builtins.iter explicitly)
_chunk_iter = builtins.iter(stream_text_chunks(TEXT_PATH))

def refill_ring_to(target_tokens):
    """Keep reading stream until ring.available() >= target_tokens (wraps file if needed)."""
    global _chunk_iter
    while ring.available() < target_tokens:
        try:
            s = next(_chunk_iter)
        except StopIteration:
            _chunk_iter = builtins.iter(stream_text_chunks(TEXT_PATH))
            s = next(_chunk_iter)
        ring.add_many(encode(s + "\n"))

# Ensure we can sample, then fill the ring all the way
if ring.available() < WARMUP_TOKENS:
    refill_ring_to(WARMUP_TOKENS)
refill_ring_to(RING_TOKENS_CAP)

# Finalize validation buffer (DO NOT shuffle token order)
val_arr = np.array(val_tokens, dtype=np.int64); del val_tokens

print(f"val buffer tokens: {len(val_arr):,} ; ring filled: {ring.available():,}")

def get_val_batch():
    hi = len(val_arr) - (block_size + 1)
    idxs = np.random.randint(0, hi + 1, size=batch_size, dtype=np.int64)
    x = np.stack([val_arr[s:s+block_size] for s in idxs], axis=0)
    y = np.stack([val_arr[s+1:s+block_size+1] for s in idxs], axis=0)
    return torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)

def top_up_ring(min_add_tokens=TOKENS_PER_TOPUP):
    """Aggressively add tokens to keep training pool fresh."""
    global _chunk_iter
    added = 0
    while added < min_add_tokens:
        try:
            s = next(_chunk_iter)
        except StopIteration:
            _chunk_iter = builtins.iter(stream_text_chunks(TEXT_PATH))
            s = next(_chunk_iter)
        ids = encode(s + "\n")
        if not ids:
            continue
        ring.add_many(ids)
        added += len(ids)

# --------------------
# Model (GPT-like)
# --------------------
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x); q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1); wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj  = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd_):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd_, 4 * n_embd_),
            nn.ReLU(),
            nn.Linear(4 * n_embd_, n_embd_),
            nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd_, n_head_):
        super().__init__()
        head_size = n_embd_ // n_head_
        self.sa = MultiHeadAttention(n_head_, head_size)
        self.ffwd = FeedForward(n_embd_)
        self.ln1 = nn.LayerNorm(n_embd_)
        self.ln2 = nn.LayerNorm(n_embd_)
    def forward(self, x):
        y = self.sa(x); x = self.ln1(x + y)
        y = self.ffwd(x); x = self.ln2(x + y)
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size_):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table    = nn.Embedding(vocab_size_, n_embd)
        self.position_embedding_table = nn.Embedding(self.block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size_, bias=False)
        # Init
        self.apply(self._init_weights)
        # Tie weights
        self.lm_head.weight = self.token_embedding_table.weight

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if getattr(m, 'bias', None) is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        if index.size(1) > self.block_size:
            index = index[:, -self.block_size:]
        B, T = index.shape
        tok = self.token_embedding_table(index)
        pos = torch.arange(T, device=index.device)
        pos_emb = self.position_embedding_table(pos)
        x = tok + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            if targets.size(1) > self.block_size:
                targets = targets[:, -self.block_size:]
            B2, T2, C = logits.shape
            loss = F.cross_entropy(logits.view(B2*T2, C), targets.view(B2*T2))
        return logits, loss

    @torch.no_grad()
    def generate(self, index, max_new_tokens):
        self.eval()
        idx = index
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            nxt = torch.multinomial(probs, 1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

# --------------------
# Meta + saving
# --------------------
def build_meta():
    return {
        'stoi': string_to_int,
        'itos': int_to_string,
        'vocab_size': len(string_to_int),
        'block_size': block_size,
        'n_embd': n_embd, 'n_head': n_head, 'n_layer': n_layer,
    }

def atomic_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

def save_ckpt(path, iter_idx, best_val, model_state, ema_state, optimizer_state, scheduler_state):
    atomic_save({
        'model_state': model_state,
        'ema_state': ema_state,
        'optimizer_state': optimizer_state,
        'scheduler_state': scheduler_state,
        'iter': iter_idx,
        'best_val_loss': best_val,
        'meta': build_meta(),
    }, path)

# --------------------
# Init model/EMA/opt, optional resume (weights later)
# --------------------
model = GPTLanguageModel(vocab_size).to(device)
ema_model = GPTLanguageModel(vocab_size).to(device)
ema_model.load_state_dict(model.state_dict())
for p in ema_model.parameters():
    p.requires_grad_(False)

# Weight decay: exclude biases & LayerNorm from WD
decay, no_decay, seen = [], [], set()
for n, p in model.named_parameters():
    if id(p) in seen:
        continue
    seen.add(id(p))
    if p.ndim == 1 or n.endswith('.bias'):
        no_decay.append(p)
    else:
        decay.append(p)

optimizer = torch.optim.AdamW(
    [{'params': decay, 'weight_decay': WEIGHT_DECAY},
     {'params': no_decay, 'weight_decay': 0.0}],
    lr=learning_rate
)

# --- AMP setup (CUDA only) ---
use_amp = (device == 'cuda')
amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
# New API (no deprecation warning). GradScaler is useful for fp16 but not bf16.
scaler = torch.amp.GradScaler("cuda", enabled=use_amp and amp_dtype is torch.float16)

# LR schedule (warmup + cosine), respecting call order (scheduler.step AFTER optimizer.step)
def lr_lambda(step):
    step = step + 1  # because we call scheduler.step() after optimizer.step()
    if step <= WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    p = (step - WARMUP_STEPS) / max(1, max_iters - WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * p))

start_iter = 0
best_val_loss = float('inf')

# Helper to strip compile prefix from keys in checkpoints saved after torch.compile
def _strip_orig_mod(sd):
    return { (k.replace('_orig_mod.', '') if k.startswith('_orig_mod.') else k): v
             for k, v in sd.items() }

# If full ckpt exists, resume
ckpt = None
try:
    ckpt = torch.load(CKPT_PATH, map_location=device)
except FileNotFoundError:
    ckpt = None

if ckpt is not None:
    try:
        msd = _strip_orig_mod(ckpt['model_state'])
        esd = _strip_orig_mod(ckpt.get('ema_state', ckpt['model_state']))
        model.load_state_dict(msd, strict=True)
        ema_model.load_state_dict(esd, strict=True)
        optimizer.load_state_dict(ckpt['optimizer_state'])
        start_iter = ckpt.get('iter', 0)
        best_val_loss = ckpt.get('best_val_loss', float('inf'))
        print(f"Resumed from {CKPT_PATH} @ iter={start_iter}, best_val={best_val_loss:.4f}")
    except Exception as e:
        print(f"Warning: failed to load weights/optimizer from checkpoint: {e}")
        ema_model.load_state_dict(model.state_dict())

# Scheduler created after knowing start_iter, then optionally load state
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda, last_epoch=start_iter - 1
)
if ckpt is not None and 'scheduler_state' in ckpt:
    try:
        scheduler.load_state_dict(ckpt['scheduler_state'])
    except Exception as e:
        print(f"Warning: failed to load scheduler state: {e}")

# Optional: compile for speed (PyTorch 2.x)
try:
    model = torch.compile(model)
except Exception:
    pass

# --------------------
# Eval (uses EMA)
# --------------------
@torch.no_grad()
def estimate_loss(eval_model):
    prev_mode = eval_model.training
    eval_model.eval()
    try:
        out = {}
        # train loss from ring
        tl = []
        for _ in range(eval_iters):
            X, Y = ring.sample_batch(batch_size, block_size)
            with torch.autocast(device_type='cuda', dtype=amp_dtype, enabled=use_amp):
                _, loss = eval_model(X, Y)
            tl.append(loss.item())
        out['train'] = float(np.mean(tl))
        # val loss from fixed buffer
        vl = []
        for _ in range(eval_iters):
            X, Y = get_val_batch()
            with torch.autocast(device_type='cuda', dtype=amp_dtype, enabled=use_amp):
                _, loss = eval_model(X, Y)
            vl.append(loss.item())
        out['val'] = float(np.mean(vl))
        return out
    finally:
        eval_model.train(prev_mode)

def ema_update():
    with torch.no_grad():
        msd = model.state_dict()
        esd = ema_model.state_dict()
        for k in msd.keys():
            # Handle potential _orig_mod prefix if the main model was compiled
            if k.startswith('_orig_mod.'):
                esd[k[len('_orig_mod.'):]].mul_(EMA_DECAY).add_(msd[k], alpha=1.0 - EMA_DECAY)
            else:
                esd[k].mul_(EMA_DECAY).add_(msd[k], alpha=1.0 - EMA_DECAY)

# --------------------
# Training (grad accumulation + AMP; patience-only early stop)
# --------------------
no_improve = 0
final_loss = None
last_iter = start_iter

for step in range(start_iter, max_iters):
    last_iter = step

    # Keep training pool fresh
    top_up_ring()

    # Periodic eval with EMA weights
    if step % eval_iters == 0:
        lr_now = scheduler.get_last_lr()[0]
        losses = estimate_loss(ema_model)
        train_loss = losses['train']; val_loss = losses['val']
        diff = train_loss - val_loss
        bpc_train = train_loss / math.log(2)
        bpc_val   = val_loss   / math.log(2)
        print(f"step: {step}, lr {lr_now:.6e}, "
              f"train {train_loss:.3f} (bpc {bpc_train:.3f})  "
              f"val {val_loss:.3f} (bpc {bpc_val:.3f})  "
              f"(diff {diff:+.3f})")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_ckpt(CKPT_PATH, step, best_val_loss,
                      model.state_dict(), ema_model.state_dict(),
                      optimizer.state_dict(), scheduler.state_dict())
            no_improve = 0
            print(f"✅ Saved new best (EMA) with meta: val {best_val_loss:.4f}")
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"⏹️ Early stop: no validation improvement for {PATIENCE} evals.")
            break

    # ---- Train step with gradient accumulation ----
    optimizer.zero_grad(set_to_none=True)
    loss_accum = 0.0

    for _ in range(GRAD_ACCUM_STEPS):
        xb, yb = ring.sample_batch(batch_size, block_size)
        with torch.autocast(device_type='cuda', dtype=amp_dtype, enabled=use_amp):
            _, loss = model(xb, yb)
            loss = loss / GRAD_ACCUM_STEPS  # normalize per micro-step

        if scaler is not None and scaler.is_enabled():
            scaler.scale(loss).backward()
        else:
            loss.backward()

        loss_accum += float(loss.item())

    # Unscale (if needed), clip, step
    if scaler is not None and scaler.is_enabled():
        scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

    if scaler is not None and scaler.is_enabled():
        scaler.step(optimizer)
        scaler.update()
    else:
        optimizer.step()

    scheduler.step()
    ema_update()
    final_loss = loss_accum

print("done. last minibatch (accumulated) loss:", None if final_loss is None else f"{final_loss:.4f}")
print(f"Best validation loss observed: {best_val_loss:.4f}")
print(f"Best checkpoint saved to: {CKPT_PATH}")

save_ckpt(FINAL_CKPT_PATH, last_iter, best_val_loss,
          model.state_dict(), ema_model.state_dict(),
          optimizer.state_dict(), scheduler.state_dict())
print(f"Final checkpoint (EMA + meta) saved to: {FINAL_CKPT_PATH}")

# --------------------
# Text generation (reload best EMA)
# --------------------
with torch.no_grad():
    try:
        ckpt = torch.load(CKPT_PATH, map_location=device)
    except FileNotFoundError:
        ckpt = torch.load(FINAL_CKPT_PATH, map_location=device)

    meta = ckpt['meta']
    string_to_int = meta['stoi']; int_to_string = meta['itos']

    # Rebuild the model for generation if vocab size changed
    if 'vocab_size' in meta and vocab_size != meta['vocab_size']:
        vocab_size = meta['vocab_size']
    gen_model = GPTLanguageModel(vocab_size).to(device)

    # Clean any _orig_mod prefix from compiled checkpoints
    state = ckpt.get('ema_state', ckpt['model_state'])
    cleaned_state = {k.replace('_orig_mod.', ''): v for k, v in state.items()}
    gen_model.load_state_dict(cleaned_state)
    gen_model.eval()
    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    print(decode(gen_model.generate(context, max_new_tokens=500)[0].tolist()))


device: cuda
Loaded tokenizer from drive/MyDrive/txt data/models/plaintext-wikipedia.pt (vocab_size=96)
vocab_size: 96
val buffer tokens: 5,000,000 ; ring filled: 500,000,000
Resumed from drive/MyDrive/txt data/models/plaintext-wikipedia.pt @ iter=36300, best_val=0.8620
step: 36300, lr 2.456848e-04, train 0.834 (bpc 1.203)  val 0.871 (bpc 1.257)  (diff -0.037)


W1111 13:31:34.022000 442 torch/_inductor/utils.py:1436] [0/0] Not enough SMs to use max_autotune_gemm mode


step: 36450, lr 2.456461e-04, train 0.865 (bpc 1.248)  val 0.866 (bpc 1.250)  (diff -0.001)
step: 36600, lr 2.456072e-04, train 0.866 (bpc 1.249)  val 0.869 (bpc 1.253)  (diff -0.003)
step: 36750, lr 2.455681e-04, train 0.857 (bpc 1.236)  val 0.863 (bpc 1.245)  (diff -0.006)
step: 36900, lr 2.455289e-04, train 0.850 (bpc 1.226)  val 0.868 (bpc 1.252)  (diff -0.018)
step: 37050, lr 2.454895e-04, train 0.849 (bpc 1.224)  val 0.861 (bpc 1.242)  (diff -0.012)
✅ Saved new best (EMA) with meta: val 0.8610
step: 37200, lr 2.454499e-04, train 0.850 (bpc 1.226)  val 0.870 (bpc 1.255)  (diff -0.020)
step: 37350, lr 2.454101e-04, train 0.842 (bpc 1.214)  val 0.871 (bpc 1.256)  (diff -0.029)
step: 37500, lr 2.453702e-04, train 0.841 (bpc 1.214)  val 0.870 (bpc 1.256)  (diff -0.029)
step: 37650, lr 2.453301e-04, train 0.861 (bpc 1.243)  val 0.866 (bpc 1.250)  (diff -0.005)
step: 37800, lr 2.452898e-04, train 0.865 (bpc 1.248)  val 0.861 (bpc 1.242)  (diff +0.004)
step: 37950, lr 2.452494e-04, train